# Among Us — Theory-of-Mind Evaluation (Qwen-3-4B)

End-to-end pipeline: run games with epistemic state collection enabled,
then compute the full suite of Theory-of-Mind metrics.

**Pipeline:**
1. **Setup** — imports, paths, vLLM server
2. **Configuration** — model, game config, agent config
3. **Launch vLLM & Patch Agent** — local model serving
4. **Run Games** — play N games with epistemic collection enabled
5. **Compute ToM Metrics** — parse epistemic logs, evaluate all 8 metrics
6. **Results Table** — display the consolidated metrics DataFrame

---
## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
import re
import asyncio
import datetime
import subprocess
import random
import time
import signal

import numpy as np
import pandas as pd
import aiohttp

from dotenv import load_dotenv
# Primary .env from project root (if any), then the per-user secrets file
# that holds OPENAI_API_KEY and ANTHROPIC_API_KEY for closed-model evals.
load_dotenv()
_SECRETS_ENV = "/home/kmirakho/.envfile.karan.tom_for_strategic_games"
if os.path.exists(_SECRETS_ENV):
    load_dotenv(_SECRETS_ENV, override=False)
    print(f"Loaded API keys from {_SECRETS_ENV}")
print(f"  OPENAI_API_KEY    set: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"  ANTHROPIC_API_KEY set: {bool(os.getenv('ANTHROPIC_API_KEY'))}")

# ── Project paths ──────────────────────────────────────────────
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
AMONG_AGENTS_PATH = os.path.join(ROOT_PATH, "among-agents")
# All evaluation artifacts (raw game logs, parsed CSVs, calibration
# figures, sweep summaries) are written under this single tree.
EVAL_BASE_PATH = "/data/kmirakho/eval-among-us"
LOGS_PATH = EVAL_BASE_PATH
RESULTS_PATH = os.path.join(EVAL_BASE_PATH, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(LOGS_PATH, exist_ok=True)

sys.path.insert(0, AMONG_AGENTS_PATH)
sys.path.insert(0, ROOT_PATH)

print(f"Root path:         {ROOT_PATH}")
print(f"Among-agents path: {AMONG_AGENTS_PATH}")
print(f"Logs path:         {LOGS_PATH}")
print(f"Results path:      {RESULTS_PATH}")

Loaded API keys from /home/kmirakho/.envfile.karan.tom_for_strategic_games
  OPENAI_API_KEY    set: True
  ANTHROPIC_API_KEY set: True
Root path:         /home/kmirakho/Documents/among-us
Among-agents path: /home/kmirakho/Documents/among-us/among-agents
Logs path:         /data/kmirakho/eval-among-us
Results path:      /data/kmirakho/eval-among-us/results


In [2]:
from amongagents.envs.game import AmongUs
from amongagents.envs.configs.game_config import (
    FIVE_MEMBER_GAME,
    THREE_MEMBER_GAME,
    SEVEN_MEMBER_GAME,
)
from utils import setup_experiment, load_agent_logs_df, load_game_summary
from evaluations.metrics_calculator import (
    EpistemicLogParser,
    TheoryOfMindEvaluator,
    GameGroundTruth,
    process_experiment,
    _build_shortest_paths,
    _extract_ground_truth_from_summary,
    _extract_regions_from_logs,
    _extract_survival_and_tasks,
    _load_summary,
)

print("All imports successful.")

All imports successful.


---
## 2. Configuration

In [ ]:
# ── GPU pool ──────────────────────────────────────────────────
# This notebook itself runs no on-GPU compute — it just orchestrates
# HTTP calls to vLLM. Each vLLM server is pinned to its own GPU(s)
# via the per-entry ``gpu`` field below; the launcher in cell 7 sets
# ``CUDA_VISIBLE_DEVICES`` per *subprocess* so models stay isolated
# and can run concurrently on a multi-GPU host.
#
# ``gpu`` is either a single id ("3") or a comma-separated list for
# tensor-parallel hosting ("3,5" → vLLM gets ``--tensor-parallel-size 2``,
# weights split across the two GPUs). Inspect availability with
# ``nvidia-smi`` before launching cell 7 and update assignments below
# if other jobs are using your idle GPUs.

# ── Local (vLLM-served) models ────────────────────────────────
VLLM_HOST = "0.0.0.0"

VLLM_LOCAL_MODELS: "dict[str, dict]" = {
    "qwen3-4b": {
        "model_path": "/data/kmirakho/verl/models/Qwen3-4B-Instruct-2507",
        "port": 8234,
        "gpu": "3",  # idle on this host (nvidia-smi 2026-04-26)
        # Non-reasoning model: 1024 is plenty for action JSON + a short
        # reasoning_scratchpad on the epistemic-state extraction call.
        "max_tokens": 1024,
    },
    # DeepSeek-R1 reasoning distilled onto a Llama-3.1 backbone. The 8B
    # variant fits comfortably in a single 49 GB RTX 6000 Ada at fp16;
    # switch to the 70B entry below for multi-GPU tensor-parallel.
    "deepseek-r1-distill-llama-8b": {
        "model_path": "/data/kmirakho/verl/models/DeepSeek-R1-Distill-Llama-8B",
        "port": 8235,
        "gpu": "5",
        # Reasoning model — emits long <think>...</think> chains BEFORE
        # the visible JSON answer. The previous notebook-wide cap of
        # 512 was eaten by <think> tokens so the JSON never made it
        # out, and `_parse_epistemic_json` silently dropped the
        # snapshot (see 2026-04-26_exp_2/3 — only 1/3 valid snapshots
        # out of ~50 expected). 1024 gives just enough headroom for a
        # brief CoT plus the belief_distribution payload.
        "max_tokens": 1024,
    },
    "llama-3.1-8b-instruct": {
        "model_path": "/data/kmirakho/verl/models/Llama-3.1-8B-Instruct",
        "port": 8236,
        "gpu": "7",
        "max_tokens": 1024,
    },
    # ── Larger Llama-family variants (commented for future use) ────
    # 70B fp16 needs ~140 GB VRAM → 3-way tensor-parallel across our
    # 49 GB cards. Re-enable once the weights are downloaded; the
    # launcher will pass ``--tensor-parallel-size`` automatically
    # based on the comma count in ``gpu``.
    # "deepseek-r1-distill-llama-70b": {
    #     "model_path": "/data/kmirakho/verl/models/DeepSeek-R1-Distill-Llama-70B",
    #     "port": 8237,
    #     "gpu": "3,5,7",
    # },
    # "llama-3.3-70b-instruct": {
    #     "model_path": "/data/kmirakho/verl/models/Llama-3.3-70B-Instruct",
    #     "port": 8238,
    #     "gpu": "3,5,7",
    # },
}

def _vllm_url(port: int) -> str:
    return f"http://localhost:{port}/v1/chat/completions"

# ── Back-compat aliases ───────────────────────────────────────
# A handful of older cells (legacy logprobs path, kill-server cell)
# still reference the singular `VLLM_API_URL` / `VLLM_PORT` /
# `LOCAL_MODEL_PATH` / `QWEN_MODEL`. Point them at the *first* vLLM
# entry above so those code paths keep working unchanged; per-agent
# routing happens via `self.vllm_url` in cell 8.
_PRIMARY_VLLM_KEY = next(iter(VLLM_LOCAL_MODELS))
LOCAL_MODEL_PATH = VLLM_LOCAL_MODELS[_PRIMARY_VLLM_KEY]["model_path"]
VLLM_PORT = VLLM_LOCAL_MODELS[_PRIMARY_VLLM_KEY]["port"]
VLLM_API_URL = _vllm_url(VLLM_PORT)
QWEN_MODEL = LOCAL_MODEL_PATH

# ── Multi-configuration sweep settings ────────────────────────
NUM_GAMES_PER_CONFIG = 10
RATE_LIMIT = 20
# Toggle: set to False to skip the second LLM call per snapshot that
# elicits token-logprob belief distributions. With it off we still
# get verbalized beliefs/votes; we just don't collect the
# logprob-derived signal (faster + cheaper). Cell 9 reads this flag.
LOGPROBS_ENABLED = False

# ── Model backends ────────────────────────────────────────────
# Each entry maps a human-readable key -> backend + model_id + concurrency.
# The agent's ``self.model`` attribute is set to the key (used as a tag in
# experiment directories and DataFrames); routing is done by ``self.backend``
# in the patched send_request (cell 8); for vLLM models, ``self.vllm_url``
# is also bound there from the per-model ``vllm_url`` field below.
#
# Active sweep (this run): 3 locally-hosted + 2 OpenAI API
#   - qwen3-4b                       : Qwen-3 4B local vLLM baseline
#   - deepseek-r1-distill-llama-8b   : DeepSeek-R1 reasoning distilled
#                                      onto a Llama-3.1 8B backbone
#   - llama-3.1-8b-instruct          : Meta's Llama-3.1 8B Instruct
#   - gpt-5.4-nano / -mini           : OpenAI reasoning family @ medium
#
# Inactive (kept defined but commented out of MODELS_TO_RUN below):
#   - gpt-5.4 (full)                 : cost-gated
#   - claude-haiku-4-5 / sonnet-4-6  : out-of-budget for this sweep
#   - 70B Llama-family variants      : need multi-GPU / quantization
#
# To bring any of those back online: uncomment the dict entry AND the
# corresponding line in MODELS_TO_RUN.
MODEL_BACKENDS: "dict[str, dict]" = {
    # ── Local vLLM models ───────────────────────────────────────
    "qwen3-4b": {
        "backend": "vllm",
        "model_id": VLLM_LOCAL_MODELS["qwen3-4b"]["model_path"],
        "vllm_url": _vllm_url(VLLM_LOCAL_MODELS["qwen3-4b"]["port"]),
        "rate_limit": 20,
    },
    "deepseek-r1-distill-llama-8b": {
        "backend": "vllm",
        "model_id": VLLM_LOCAL_MODELS["deepseek-r1-distill-llama-8b"]["model_path"],
        "vllm_url": _vllm_url(VLLM_LOCAL_MODELS["deepseek-r1-distill-llama-8b"]["port"]),
        "rate_limit": 16,
    },
    "llama-3.1-8b-instruct": {
        "backend": "vllm",
        "model_id": VLLM_LOCAL_MODELS["llama-3.1-8b-instruct"]["model_path"],
        "vllm_url": _vllm_url(VLLM_LOCAL_MODELS["llama-3.1-8b-instruct"]["port"]),
        "rate_limit": 16,
    },
    # ── 70B Llama-family variants (commented; uncomment alongside
    # the matching VLLM_LOCAL_MODELS entry above) ───────────────
    # "deepseek-r1-distill-llama-70b": {
    #     "backend": "vllm",
    #     "model_id": VLLM_LOCAL_MODELS["deepseek-r1-distill-llama-70b"]["model_path"],
    #     "vllm_url": _vllm_url(VLLM_LOCAL_MODELS["deepseek-r1-distill-llama-70b"]["port"]),
    #     "rate_limit": 8,
    # },
    # "llama-3.3-70b-instruct": {
    #     "backend": "vllm",
    #     "model_id": VLLM_LOCAL_MODELS["llama-3.3-70b-instruct"]["model_path"],
    #     "vllm_url": _vllm_url(VLLM_LOCAL_MODELS["llama-3.3-70b-instruct"]["port"]),
    #     "rate_limit": 8,
    # },
    # ── GPT-5.4 family (medium reasoning effort) ────────────────
    "gpt-5.4-nano": {
        "backend": "openai",
        "model_id": "gpt-5.4-nano",
        "rate_limit": 4,
        "reasoning_effort": "low",
    },
    "gpt-5.4-mini": {
        "backend": "openai",
        "model_id": "gpt-5.4-mini",
        "rate_limit": 4,
        "reasoning_effort": "medium",
    },
    # ── Cost-gated entries (kept for upgradeability) ────────────
    # "gpt-5.4": {
    #     "backend": "openai",
    #     "model_id": "gpt-5.4",
    #     "rate_limit": 4,
    #     "reasoning_effort": "medium",
    # },
    # ── Claude family (extended thinking enabled) ───────────────
    # ``thinking_budget`` caps thinking tokens per response.
    # Currently out-of-budget — re-enable both the entry below and
    # the corresponding line in MODELS_TO_RUN to bring them back.
    # "claude-haiku-4-5": {
    #     "backend": "anthropic",
    #     "model_id": "claude-haiku-4-5",
    #     "rate_limit": 3,
    #     "thinking_budget": 4096,
    # },
    # "claude-sonnet-4-6": {
    #     "backend": "anthropic",
    #     "model_id": "claude-sonnet-4-6",
    #     "rate_limit": 3,
    #     "thinking_budget": 4096,
    # },
}

# Models actually run in the sweep. Costly closed-source / oversized
# entries are commented out so this list stays in sync with the
# (commented) entries in MODEL_BACKENDS.
MODELS_TO_RUN = [
    "qwen3-4b",
    "deepseek-r1-distill-llama-8b",
    # Pending Meta access grant for meta-llama/Llama-3.1-8B-Instruct;
    # uncomment once the weights land in the configured local path.
    "llama-3.1-8b-instruct",
    "gpt-5.4-nano",
    "gpt-5.4-mini",
    # "gpt-5.4",
    # "claude-haiku-4-5",
    # "claude-sonnet-4-6",
    # "deepseek-r1-distill-llama-70b",
    # "llama-3.3-70b-instruct",
]

# ⚠️  RUNTIME / COST NOTE
# A single Among-Us game generates ~100–200 LLM calls (agents act every
# timestep + discussion turns + per-agent epistemic-state snapshots at
# each meeting). With the active sweep (4 models × 2 configs × 10 games
# = 80 games) expect:
#   - Local 8B vLLM models (GPU-bound)   : ~30–40 h serial each, much
#                                          less with vLLM batching at
#                                          high rate_limit
#   - GPT-5.4 nano / mini  (network I/O) : ~4–8 h at rate_limit=4
#   - Claude family (re-enabled)         : ~6–10 h at rate_limit=3
# Each local vLLM server is pinned to its own GPU (see ``gpu`` field
# in VLLM_LOCAL_MODELS above) and gets gpu-memory-utilization=0.85 in
# the launcher (cell 7). With one model per GPU all local servers can
# run in parallel; reduce NUM_GAMES_PER_CONFIG to 2–3 to smoke-test
# the full pipeline before the real run.

# Four (crewmate-count, impostor-count) game configurations.
# Each config is a dict compatible with AmongUs(game_config=...).
# We derive the rest from FIVE_MEMBER_GAME as a sensible baseline and
# override num_players / num_impostors for the sweep.
def _make_cfg(n_crew: int, n_imp: int, base: dict) -> dict:
    cfg = dict(base)
    cfg["num_players"] = n_crew + n_imp
    cfg["num_impostors"] = n_imp
    return cfg

GAME_CONFIGS: "dict[str, dict]" = {
    # NOTE (one-off run): scoped to the two 5-crewmate configs only.
    # Restore "4C_1I" / "4C_2I" entries below to run the full sweep.
    # "4C_1I": _make_cfg(4, 1, FIVE_MEMBER_GAME),
    "4C_2I": _make_cfg(4, 2, FIVE_MEMBER_GAME),
    "5C_1I": _make_cfg(5, 1, FIVE_MEMBER_GAME),
    # "5C_2I": _make_cfg(5, 2, FIVE_MEMBER_GAME),
}

# ── Agent config: local Qwen-3-4B for every role ─────────────
AGENT_CONFIG = {
    "Impostor": "LLM",
    "Crewmate": "LLM",
    "CREWMATE_LLM_CHOICES": [QWEN_MODEL],
    "IMPOSTOR_LLM_CHOICES": [QWEN_MODEL],
}

# Back-compat defaults (used if any cell still references them).
GAME_CONFIG = GAME_CONFIGS["5C_1I"]
NUM_GAMES = NUM_GAMES_PER_CONFIG

GAME_ARGS = {
    "game_config": GAME_CONFIG,
    "include_human": False,
    "test": False,
    "personality": False,
    "agent_config": AGENT_CONFIG,
    "UI": False,
    "Streamlit": False,
}

# Active local vLLM models (subset of VLLM_LOCAL_MODELS that's actually
# in this sweep). The launcher in cell 7 iterates over this list, and
# cell 8's health-check uses it to ping every running server.
ACTIVE_VLLM_MODELS = [
    m for m in MODELS_TO_RUN
    if MODEL_BACKENDS[m]["backend"] == "vllm"
]

print(f"vLLM hosts            :")
for k in ACTIVE_VLLM_MODELS:
    spec = VLLM_LOCAL_MODELS[k]
    print(f"  {k:30s} -> GPU {spec['gpu']:6s} port {spec['port']}  | {spec['model_path']}")
print(f"Games per config      : {NUM_GAMES_PER_CONFIG}")
print(f"Configurations        : {list(GAME_CONFIGS.keys())}")
for name, cfg in GAME_CONFIGS.items():
    print(f"  {name}: players={cfg['num_players']}, "
          f"impostors={cfg['num_impostors']}, "
          f"max_timesteps={cfg.get('max_timesteps')}")
print(f"\nModels to benchmark   : {MODELS_TO_RUN}")
for m in MODELS_TO_RUN:
    spec = MODEL_BACKENDS[m]
    print(f"  {m:30s} -> {spec['backend']:9s} | {spec['model_id']}")
print(f"\nTotal runs            : {len(MODELS_TO_RUN)} models × "
      f"{len(GAME_CONFIGS)} configs × {NUM_GAMES_PER_CONFIG} games "
      f"= {len(MODELS_TO_RUN)*len(GAME_CONFIGS)*NUM_GAMES_PER_CONFIG} games")

---
## 3. Launch vLLM Server & Patch Agent

In [ ]:
import socket
import threading
import collections

# Each vLLM subprocess streams its (chatty) request log here so it never
# ends up captured in the notebook's stream-output (which previously
# bloated the .ipynb past Cursor's 50 MB open-file cap).
VLLM_LOG_DIR = os.path.join(LOGS_PATH, "vllm-logs")
os.makedirs(VLLM_LOG_DIR, exist_ok=True)

def is_port_in_use(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("localhost", port)) == 0

# Per-model server handles, populated as we launch each one.
# The kill-server cell at the end of the notebook iterates over these.
vllm_processes: "dict[str, subprocess.Popen]" = {}
vllm_logs: "dict[str, list[str]]" = {}


def _read_vllm_output(model_key: str, proc, log_list, log_path: str):
    """Drain the vLLM subprocess stdout to a per-model logfile.

    We used to forward every line to the notebook's stdout, but vLLM's
    request log is so chatty that an overnight sweep ballooned the
    .ipynb to 50+ MB (well over Cursor's 50 MB open-file cap). Now
    lines go to ``log_path`` (line-buffered, so ``tail -f`` works) and
    ``log_list`` keeps only the last 200 lines in memory for the
    crash-tail dump in `_launch_vllm`.
    """
    with open(log_path, "a", buffering=1) as logf:
        for line in iter(proc.stdout.readline, ""):
            log_list.append(line)
            logf.write(line)
    proc.stdout.close()


def _launch_vllm(
    model_key: str,
    model_path: str,
    port: int,
    gpu: str,
) -> None:
    """Idempotently start a vLLM server for ``model_key`` on ``port``.

    The server is pinned to ``gpu`` (a string like ``"3"`` for a single
    device or ``"3,5,7"`` for tensor-parallel hosting) by setting
    ``CUDA_VISIBLE_DEVICES`` in the subprocess env. When more than one
    GPU is listed we also pass ``--tensor-parallel-size N`` so vLLM
    shards the model weights accordingly.

    If the port is already bound we assume a compatible server is up
    (e.g. left running from a previous notebook session) and skip.
    """
    if is_port_in_use(port):
        print(f"  [vLLM:{model_key}] port {port} in use — "
              f"assuming server already running.")
        return

    gpu_ids = [g.strip() for g in str(gpu).split(",") if g.strip()]
    n_gpus = len(gpu_ids)
    if n_gpus == 0:
        raise ValueError(
            f"vLLM model {model_key!r}: ``gpu`` field must list at least "
            f"one CUDA device id (got {gpu!r})."
        )

    cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", model_path,
        "--host", VLLM_HOST,
        "--port", str(port),
        "--dtype", "auto",
        "--max-model-len", "16384",
        # Each GPU is dedicated to one server now (pinned via
        # CUDA_VISIBLE_DEVICES below), so we can claim ~80% of device
        # memory. Was 0.85, but on shared GPU boxes other tenants
        # routinely hold 5–8 GiB of unrelated work, which makes 0.85
        # fail at boot with "Free memory on device ... less than
        # desired GPU memory utilization". 0.80 leaves ~9.5 GiB
        # headroom on a 49 GiB card.
        "--gpu-memory-utilization", "0.80",
        # Required for token-logprob-based epistemic state extraction
        # (`belief_distribution_logprobs` / `voting_intent_logprobs` request
        # top_logprobs=20 from the OpenAI-compatible endpoint).
        "--max-logprobs", "20",
    ]
    if n_gpus > 1:
        cmd.extend(["--tensor-parallel-size", str(n_gpus)])

    # Per-process GPU pinning. The vLLM worker only sees the GPUs we
    # list here, and indices are renumbered from 0 inside the process,
    # so two servers can both target "GPU 0" without colliding.
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = ",".join(gpu_ids)

    tp_msg = f" tp={n_gpus}" if n_gpus > 1 else ""
    print(f"Starting vLLM server for {model_key} on GPU(s) "
          f"{env['CUDA_VISIBLE_DEVICES']} (port {port}){tp_msg}...")

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        preexec_fn=os.setsid,
        text=True,
        bufsize=1,
        env=env,
    )
    vllm_processes[model_key] = proc
    vllm_logs[model_key] = collections.deque(maxlen=200)

    log_path = os.path.join(VLLM_LOG_DIR, f"{model_key}.log")
    log_thread = threading.Thread(
        target=_read_vllm_output,
        args=(model_key, proc, vllm_logs[model_key], log_path),
        daemon=True,
    )
    log_thread.start()
    print(f"  [vLLM:{model_key}] launched (PID: {proc.pid}). "
          f"streaming server log -> {log_path}")
    print(f"  [vLLM:{model_key}] waiting for ready...")

    for _ in range(300):
        time.sleep(2)
        if proc.poll() is not None:
            print(f"\n*** [vLLM:{model_key}] exited with code {proc.returncode} ***")
            # ``vllm_logs[model_key]`` is a ``collections.deque`` —
            # deques do NOT support slice indexing, so coerce to list
            # for the tail dump (otherwise this diagnostic raises
            # TypeError and hides the real crash output).
            log_tail = list(vllm_logs[model_key])[-20:]
            for line in log_tail:
                print(f"  {line}", end="")
            raise RuntimeError(
                f"vLLM server for {model_key} crashed "
                f"(exit code {proc.returncode})."
            )
        if is_port_in_use(port):
            print(f"  [vLLM:{model_key}] ready on port {port}!")
            return

    raise RuntimeError(
        f"vLLM server for {model_key} did not start within 600s."
    )


if ACTIVE_VLLM_MODELS:
    # Sanity-check: no two active models share a GPU id (would OOM
    # immediately since each claims 0.85 of device memory).
    _gpu_owner: "dict[str, str]" = {}
    for _key in ACTIVE_VLLM_MODELS:
        for _gid in str(VLLM_LOCAL_MODELS[_key]["gpu"]).split(","):
            _gid = _gid.strip()
            if _gid in _gpu_owner:
                raise RuntimeError(
                    f"GPU {_gid} is assigned to both "
                    f"{_gpu_owner[_gid]!r} and {_key!r} — update the "
                    f"``gpu`` fields in VLLM_LOCAL_MODELS so each "
                    f"active model has its own device(s)."
                )
            _gpu_owner[_gid] = _key

    for _key in ACTIVE_VLLM_MODELS:
        _spec = VLLM_LOCAL_MODELS[_key]
        _launch_vllm(_key, _spec["model_path"], _spec["port"], _spec["gpu"])
else:
    print("No vLLM-backed models in MODELS_TO_RUN — skipping local server launch.")

# Back-compat aliases: legacy cells (kill-server / introspection) still
# reference singular ``vllm_process`` / ``vllm_log_lines``. Expose the
# *primary* server's handles so those code paths continue to work.
vllm_process = vllm_processes.get(_PRIMARY_VLLM_KEY)
# ``vllm_logs[...]`` is a deque (no slicing). Coerce to list so legacy
# callers that do ``vllm_log_lines[-N:]`` keep working.
vllm_log_lines = list(vllm_logs.get(_PRIMARY_VLLM_KEY, []))

In [ ]:
from amongagents.agent.agent import LLMAgent

OPENAI_API_URL = "https://api.openai.com/v1/chat/completions"
ANTHROPIC_API_URL = "https://api.anthropic.com/v1/messages"

# ── Prompt-caching telemetry ──────────────────────────────────────────
# Both backends now exploit prompt caching:
#   * OpenAI auto-caches any prompt prefix ≥1024 tokens; we add a
#     ``prompt_cache_key`` per (model, role) so requests with the
#     identical system prompt get routed to the same cache shard.
#   * Anthropic requires explicit ``cache_control: ephemeral`` markers;
#     we attach one to the (large, static) system prompt block so the
#     5-minute cache absorbs ~95% of input tokens after the first call.
# Cached input is ~10× cheaper than fresh input on both providers; the
# counters below let you verify cache-hit rates after a sweep.
TOKEN_USAGE: "dict[str, dict[str, int]]" = {}


def _record_usage(
    model_key: str,
    in_tok: int,
    out_tok: int,
    cached_tok: int = 0,
    reasoning_tok: int = 0,
) -> None:
    u = TOKEN_USAGE.setdefault(
        model_key,
        {"calls": 0, "in": 0, "out": 0, "cached_in": 0, "reasoning": 0},
    )
    u["calls"] += 1
    u["in"] += int(in_tok)
    u["out"] += int(out_tok)
    u["cached_in"] += int(cached_tok)
    u["reasoning"] += int(reasoning_tok)


async def _send_vllm(self, messages: list) -> str:
    # Per-agent URL (set by `_patched_llmagent_init` from the
    # ``vllm_url`` field in MODEL_BACKENDS) routes each request to the
    # vLLM server hosting *this* agent's model. Falls back to the
    # primary server's URL for legacy code paths that pre-date the
    # multi-vLLM refactor.
    url = getattr(self, "vllm_url", None) or VLLM_API_URL
    # Per-model max_tokens (bound in `_patched_llmagent_init` from the
    # ``max_tokens`` field in VLLM_LOCAL_MODELS). Reasoning models like
    # DeepSeek-R1-distill need 1024+; non-reasoning models are fine at
    # 1024 too. The conservative 512 default is only used if a model
    # spec forgot to set the field.
    max_tokens = getattr(self, "max_tokens", None) or 1024
    payload = {
        "model": self.model_id,
        "messages": messages,
        "temperature": self.temperature,
        "max_tokens": max_tokens,
        "top_p": 1,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    async with aiohttp.ClientSession() as session:
        for attempt in range(10):
            try:
                async with session.post(
                    url, headers={}, json=payload
                ) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if data.get("choices"):
                            return data["choices"][0]["message"]["content"]
                    print(f"  [vLLM] status={resp.status} retry {attempt+1}/10")
            except Exception as e:
                print(f"  [vLLM] error: {e} retry {attempt+1}/10")
            await asyncio.sleep(0.5 * 2 ** attempt)
    return ""


async def _send_openai(self, messages: list) -> str:
    """OpenAI chat/completions with GPT-5 family quirk handling.

    GPT-5 reasoning models (gpt-5.x, o-series):
      * use ``max_completion_tokens`` instead of ``max_tokens``
      * only accept ``temperature=1`` (the default); custom temperatures
        are rejected, so we omit the field
      * accept a ``reasoning_effort`` knob ("minimal" | "low" | "medium"
        | "high") configured per-model in MODEL_BACKENDS
    Older GPT-4-class models fall through to the legacy parameter set.
    """
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("  [OpenAI] OPENAI_API_KEY not set")
        return ""

    model_id = self.model_id
    model_key = getattr(self, "model", model_id)
    is_reasoning = model_id.startswith(("gpt-5", "o1", "o3", "o4"))

    payload: dict = {"model": model_id, "messages": messages}
    # Prompt-cache routing key: stable per (model, player-role) so every
    # call from the same agent re-uses the same cache shard. OpenAI
    # caches the longest matching prefix automatically once it exceeds
    # 1024 tokens — our system prompt is ~2K tokens, so essentially
    # every call after the first hits the cache.
    cache_key = f"{model_key}|{getattr(self, 'player', None) and self.player.identity or 'agent'}"
    payload["prompt_cache_key"] = cache_key
    if is_reasoning:
        # Reasoning models burn most of the budget on hidden CoT before
        # emitting visible content. 2048 was too tight for nano on the
        # epistemic-state call (large JSON output) — it routinely hit
        # max_completion_tokens during hidden reasoning and returned
        # empty content, which the wrapper then masked as "SPEAK: ..."
        # and the epistemic parser dropped the snapshot. 4096 leaves
        # ~1-2K tokens of headroom for visible output.
        payload["max_completion_tokens"] = 4096
        spec = MODEL_BACKENDS.get(model_key, {})
        effort = spec.get("reasoning_effort")
        if effort:
            payload["reasoning_effort"] = effort
    else:
        payload["max_tokens"] = 1024
        payload["temperature"] = self.temperature

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    async with aiohttp.ClientSession() as session:
        for attempt in range(8):
            try:
                async with session.post(
                    OPENAI_API_URL, headers=headers, json=payload,
                    timeout=aiohttp.ClientTimeout(total=180),
                ) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        if data.get("choices"):
                            usage = data.get("usage", {}) or {}
                            in_tok = int(usage.get("prompt_tokens", 0))
                            out_tok = int(usage.get("completion_tokens", 0))
                            cached_tok = int(
                                (usage.get("prompt_tokens_details") or {})
                                .get("cached_tokens", 0)
                            )
                            reasoning_tok = int(
                                (usage.get("completion_tokens_details") or {})
                                .get("reasoning_tokens", 0)
                            )
                            _record_usage(
                                model_key, in_tok, out_tok,
                                cached_tok, reasoning_tok,
                            )
                            return data["choices"][0]["message"]["content"]
                    body = await resp.text()
                    print(f"  [OpenAI] {model_id} status={resp.status} "
                          f"retry {attempt+1}/8: {body[:200]}")
            except Exception as e:
                print(f"  [OpenAI] error: {e} retry {attempt+1}/8")
            await asyncio.sleep(1.0 * 2 ** attempt)
    return ""


async def _send_anthropic(self, messages: list) -> str:
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        print("  [Anthropic] ANTHROPIC_API_KEY not set")
        return ""

    sys_parts, conv = [], []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")
        if role == "system":
            sys_parts.append(content)
        elif role in ("user", "assistant"):
            conv.append({"role": role, "content": content})
    if not conv or conv[0]["role"] != "user":
        conv.insert(0, {"role": "user", "content": "Proceed."})

    # ── Prompt caching ────────────────────────────────────────────────
    # Anthropic caches blocks marked with ``cache_control``. The system
    # prompt is large (~2K tokens) and stable per role, so we mark it
    # ephemeral (5-minute TTL). First call writes the cache (1.25× input
    # cost), subsequent calls hit it (0.1× input cost). For our sweep
    # rate (~3 concurrent, hundreds of calls/min) virtually every call
    # past the first one of each role is a cache hit.
    sys_text = "\n\n".join(sys_parts) if sys_parts else ""
    if sys_text:
        # Anthropic accepts ``system`` as either a string OR a list of
        # content blocks. The list form is required to attach cache_control.
        system_field: list | str = [
            {
                "type": "text",
                "text": sys_text,
                "cache_control": {"type": "ephemeral"},
            }
        ]
    else:
        system_field = ""

    model_key = getattr(self, "model", self.model_id)
    spec = MODEL_BACKENDS.get(model_key, {})
    thinking_budget = int(spec.get("thinking_budget", 0) or 0)
    # Anthropic requires max_tokens > thinking_budget; also temperature
        # must be 1.0 (or omitted) when extended thinking is enabled.
    if thinking_budget > 0:
        max_tok = thinking_budget + 1024
        payload = {
            "model": self.model_id,
            "system": system_field,
            "messages": conv,
            "max_tokens": max_tok,
            "thinking": {
                "type": "enabled",
                "budget_tokens": thinking_budget,
            },
        }
    else:
        payload = {
            "model": self.model_id,
            "system": system_field,
            "messages": conv,
            "temperature": self.temperature,
            "max_tokens": 1024,
        }
    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
        "Content-Type": "application/json",
    }
    async with aiohttp.ClientSession() as session:
        for attempt in range(8):
            try:
                async with session.post(
                    ANTHROPIC_API_URL, headers=headers, json=payload,
                    timeout=aiohttp.ClientTimeout(total=120),
                ) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        blocks = data.get("content", []) or []
                        text = "".join(
                            b.get("text", "")
                            for b in blocks
                            if b.get("type") == "text"
                        )
                        if text:
                            usage = data.get("usage", {}) or {}
                            # ``input_tokens`` excludes cached + write
                            # tokens — they're billed separately.
                            in_tok = int(usage.get("input_tokens", 0))
                            cache_read = int(
                                usage.get("cache_read_input_tokens", 0)
                            )
                            cache_write = int(
                                usage.get("cache_creation_input_tokens", 0)
                            )
                            out_tok = int(usage.get("output_tokens", 0))
                            # Lump fresh + cache-write into ``in`` so the
                            # totals reflect billable input tokens; track
                            # cache reads separately.
                            _record_usage(
                                model_key,
                                in_tok=in_tok + cache_write,
                                out_tok=out_tok,
                                cached_tok=cache_read,
                            )
                            return text
                    body = await resp.text()
                    print(f"  [Anthropic] {self.model_id} status={resp.status} "
                          f"retry {attempt+1}/8: {body[:160]}")
            except Exception as e:
                print(f"  [Anthropic] error: {e} retry {attempt+1}/8")
            await asyncio.sleep(1.5 * 2 ** attempt)
    return ""


_BACKEND_DISPATCH = {
    "vllm": _send_vllm,
    "openai": _send_openai,
    "anthropic": _send_anthropic,
}


async def _multi_backend_send_request(self, messages):
    """Route to the correct backend based on self.backend."""
    backend = getattr(self, "backend", "vllm")
    fn = _BACKEND_DISPATCH.get(backend, _send_vllm)
    out = await fn(self, messages)
    if not out:
        return "SPEAK: ..."
    return out


LLMAgent.send_request = _multi_backend_send_request


# ── Per-task model + experiment-path binding via contextvars ─────────
# We run all models concurrently (one asyncio task per model). Module
# globals (``os.environ["EXPERIMENT_PATH"]`` and a single ACTIVE_MODEL_KEY)
# would be overwritten by whichever task ran most recently, so each task
# owns its own ContextVar binding. ContextVars are copied on
# ``asyncio.create_task``, so child game tasks inherit the parent
# model+path automatically without any explicit plumbing.
import contextvars
from amongagents.envs.game import AmongUs

_DEFAULT_MODEL = MODELS_TO_RUN[0] if MODELS_TO_RUN else "qwen3-4b"
MODEL_KEY_VAR: "contextvars.ContextVar[str]" = contextvars.ContextVar(
    "MODEL_KEY", default=_DEFAULT_MODEL
)
EXP_PATH_VAR: "contextvars.ContextVar[str]" = contextvars.ContextVar(
    "EXP_PATH", default=""
)

# Idempotent stash: re-running this cell must not capture the
# already-patched __init__ (that would cause infinite recursion the
# next time an LLMAgent is constructed).
if not hasattr(LLMAgent, "_orig_init"):
    LLMAgent._orig_init = LLMAgent.__init__
_orig_llmagent_init = LLMAgent._orig_init


def _patched_llmagent_init(self, *args, **kwargs):
    _orig_llmagent_init(self, *args, **kwargs)
    model_key = MODEL_KEY_VAR.get()
    spec = MODEL_BACKENDS[model_key]
    self.backend = spec["backend"]
    self.model_id = spec["model_id"]
    self.model = model_key
    # vLLM models carry a per-server URL so we can run multiple local
    # models concurrently on distinct ports (see cell 7). Non-vLLM
    # backends just leave it as None — the OpenAI / Anthropic senders
    # ignore this field.
    self.vllm_url = spec.get("vllm_url")
    # Per-model max_tokens for vLLM. Reasoning models (DeepSeek-R1-
    # distill, etc.) need a larger budget so their <think>...</think>
    # chain doesn't eat the entire response and leave no room for the
    # visible answer. OpenAI/Anthropic backends ignore this field.
    if spec["backend"] == "vllm":
        vllm_spec = VLLM_LOCAL_MODELS.get(model_key, {}) or {}
        self.max_tokens = int(vllm_spec.get("max_tokens", 1024))
    else:
        self.max_tokens = None
    # Override the log paths the original __init__ baked in from
    # os.environ at construction time, so concurrent tasks each write
    # to their own experiment directory.
    exp_path = EXP_PATH_VAR.get() or os.environ.get("EXPERIMENT_PATH", ".")
    self.log_path = os.path.join(exp_path, "agent-logs.json")
    self.compact_log_path = os.path.join(exp_path, "agent-logs-compact.json")


LLMAgent.__init__ = _patched_llmagent_init


# ── Patch AmongUs to bind experiment path per game instance ──────────
# game.py reads ``os.environ["EXPERIMENT_PATH"]`` inline in
# ``report_winner`` and ``_save_epistemic_log``. We capture the path on
# the instance at construction time, then briefly restore it into the
# env var around those (synchronous, no-await) writes so the original
# code paths stay untouched.
_orig_amongus_init = AmongUs.__init__
_orig_report_winner = AmongUs.report_winner
_orig_save_epistemic = AmongUs._save_epistemic_log


def _patched_amongus_init(self, *args, **kwargs):
    _orig_amongus_init(self, *args, **kwargs)
    self._exp_path = (
        EXP_PATH_VAR.get() or os.environ.get("EXPERIMENT_PATH", ".")
    )


def _with_exp_env(self, fn, *args, **kwargs):
    prev = os.environ.get("EXPERIMENT_PATH")
    os.environ["EXPERIMENT_PATH"] = getattr(self, "_exp_path", prev or ".")
    try:
        return fn(self, *args, **kwargs)
    finally:
        if prev is None:
            os.environ.pop("EXPERIMENT_PATH", None)
        else:
            os.environ["EXPERIMENT_PATH"] = prev


def _patched_report_winner(self, winner):
    return _with_exp_env(self, _orig_report_winner, winner)


def _patched_save_epistemic(self):
    return _with_exp_env(self, _orig_save_epistemic)


AmongUs.__init__ = _patched_amongus_init
AmongUs.report_winner = _patched_report_winner
AmongUs._save_epistemic_log = _patched_save_epistemic


def configure_agent_for_model(model_key: str) -> None:
    """Bind the active model in the *current* contextvar context.

    Kept for backward compatibility with sequential callers; concurrent
    callers should set ``MODEL_KEY_VAR`` directly inside each task.
    """
    MODEL_KEY_VAR.set(model_key)
    spec = MODEL_BACKENDS[model_key]
    print(f"  Agents -> backend={spec['backend']:9s} | "
          f"model_id={spec['model_id']}")


async def check_vllm_health():
    """Ping every active local vLLM server and report the served model IDs."""
    served: "dict[str, list | str]" = {}
    async with aiohttp.ClientSession() as session:
        for key in ACTIVE_VLLM_MODELS:
            port = VLLM_LOCAL_MODELS[key]["port"]
            url = f"http://localhost:{port}/v1/models"
            try:
                async with session.get(url) as resp:
                    data = await resp.json()
                    served[key] = [m["id"] for m in data.get("data", [])]
            except Exception as e:
                served[key] = f"ERROR: {e}"
    for k, v in served.items():
        print(f"  vLLM:{k:30s} -> {v}")
    return served


if ACTIVE_VLLM_MODELS:
    try:
        served_models = await check_vllm_health()
    except Exception as e:
        print(f"[WARN] vLLM health check failed: {e}")

print("\nLLMAgent patched with multi-backend dispatch (vLLM / OpenAI / Anthropic).")
print(f"Default MODEL_KEY_VAR = {MODEL_KEY_VAR.get()}")
print("AmongUs patched: per-instance EXPERIMENT_PATH binding (concurrency-safe).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Logprob backend dispatch — patches `LLMAgent.send_request_with_logprobs`
#
# The base method in agent.py uses `self.api_url` (a placeholder) for the
# request endpoint. The notebook patches `send_request` to dispatch across
# vllm/openai/anthropic, so we need to do the same for the logprob variant
# or every epistemic-state extraction will hit the dummy URL and fail with
# `[LOGPROBS ERROR] ... XXXX`.
#
# - vLLM      : supported (requires server flag `--max-logprobs 20`).
# - OpenAI    : supported (chat/completions accepts `logprobs` + `top_logprobs`,
#               EXCEPT for reasoning models — they ignore/error on logprobs,
#               so we return None for those and the agent falls back gracefully).
# - Anthropic : the public API does not expose top_logprobs, so we return
#               None and the snapshot is recorded with verbalized fields only.
# ═══════════════════════════════════════════════════════════════════════════

async def _send_vllm_logprobs(
    self,
    messages: list,
    max_tokens: int = 1,
    top_logprobs: int = 20,
    temperature: float = 1.0,
):
    # Mirror the routing in `_send_vllm` (cell 8): each agent owns a
    # `vllm_url` set from MODEL_BACKENDS, so logprob probes hit the
    # *same* server that's serving that agent's model.
    url = getattr(self, "vllm_url", None) or VLLM_API_URL
    payload = {
        "model": self.model_id,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "top_p": 1,
        "frequency_penalty": 0,
        "presence_penalty": 0,
        "logprobs": True,
        "top_logprobs": top_logprobs,
        "chat_template_kwargs": {"enable_thinking": False},
    }
    async with aiohttp.ClientSession() as session:
        for attempt in range(5):
            try:
                async with session.post(
                    url, headers={}, json=payload,
                    timeout=aiohttp.ClientTimeout(total=120),
                ) as resp:
                    if resp.status != 200:
                        body = await resp.text()
                        print(f"  [vLLM logprobs] status={resp.status} "
                              f"retry {attempt+1}/5: {body[:200]}")
                    else:
                        data = await resp.json()
                        if not data.get("choices"):
                            return None
                        lp = data["choices"][0].get("logprobs")
                        if not lp or not lp.get("content"):
                            print("  [vLLM logprobs] response missing "
                                  "`logprobs.content` — server may have been "
                                  "started without --max-logprobs.")
                            return None
                        return [t.get("top_logprobs", []) for t in lp["content"]]
            except Exception as e:
                print(f"  [vLLM logprobs] error: {e} retry {attempt+1}/5")
            await asyncio.sleep(0.5 * 2 ** attempt)
    return None


async def _send_openai_logprobs(
    self,
    messages: list,
    max_tokens: int = 1,
    top_logprobs: int = 20,
    temperature: float = 1.0,
):
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        return None

    model_id = self.model_id
    is_reasoning = model_id.startswith(("gpt-5", "o1", "o3", "o4"))
    if is_reasoning:
        return None

    payload = {
        "model": model_id,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "logprobs": True,
        "top_logprobs": min(top_logprobs, 20),
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    async with aiohttp.ClientSession() as session:
        for attempt in range(5):
            try:
                async with session.post(
                    OPENAI_API_URL, headers=headers, json=payload,
                    timeout=aiohttp.ClientTimeout(total=180),
                ) as resp:
                    if resp.status != 200:
                        body = await resp.text()
                        print(f"  [OpenAI logprobs] {model_id} "
                              f"status={resp.status} retry {attempt+1}/5: "
                              f"{body[:200]}")
                    else:
                        data = await resp.json()
                        if not data.get("choices"):
                            return None
                        lp = data["choices"][0].get("logprobs")
                        if not lp or not lp.get("content"):
                            return None
                        return [t.get("top_logprobs", []) for t in lp["content"]]
            except Exception as e:
                print(f"  [OpenAI logprobs] error: {e} retry {attempt+1}/5")
            await asyncio.sleep(1.0 * 2 ** attempt)
    return None


async def _send_anthropic_logprobs(self, messages: list, **kwargs):
    # Anthropic's public API does not expose top_logprobs.
    return None


_LOGPROB_DISPATCH = {
    "vllm":      _send_vllm_logprobs,
    "openai":    _send_openai_logprobs,
    "anthropic": _send_anthropic_logprobs,
}


async def _multi_backend_send_logprobs(
    self,
    messages,
    max_tokens: int = 1,
    top_logprobs: int = 20,
    temperature: float = 1.0,
):
    """Route logprob requests to the per-backend implementation.

    Mirrors the structure of `_multi_backend_send_request` so the epistemic
    extractor in agent.py works unchanged regardless of which model is active.

    Globally bypassed when ``LOGPROBS_ENABLED`` is False — we just return
    None so the caller in agent.py records an empty logprob distribution
    (the verbalized signal is unaffected).
    """
    if not globals().get("LOGPROBS_ENABLED", True):
        return None
    backend = getattr(self, "backend", "vllm")
    fn = _LOGPROB_DISPATCH.get(backend, _send_vllm_logprobs)
    return await fn(
        self,
        messages,
        max_tokens=max_tokens,
        top_logprobs=top_logprobs,
        temperature=temperature,
    )


LLMAgent.send_request_with_logprobs = _multi_backend_send_logprobs

print("LLMAgent.send_request_with_logprobs patched for multi-backend dispatch "
      "(vllm | openai | anthropic).")


---
## 4. Run Games with Epistemic State Collection

In [ ]:
DATE = datetime.datetime.now().strftime("%Y-%m-%d")
try:
    COMMIT_HASH = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=ROOT_PATH
    ).strip().decode("utf-8")
except Exception:
    COMMIT_HASH = "unknown"

print(f"Date        : {DATE}")
print(f"Commit hash : {COMMIT_HASH}")
print(f"Logs path   : {LOGS_PATH}")
print("Each configuration will get its own experiment directory via "
      "setup_experiment() inside the sweep loop below.")

In [ ]:
# ── Concurrent sweep: one asyncio task per model, all running together ──
# Each model task owns its own copies of MODEL_KEY_VAR and EXP_PATH_VAR
# (ContextVars are cloned per-task), so vLLM/OpenAI/Anthropic games can
# all be in-flight simultaneously without trampling each other's
# experiment directories or backend bindings.
#
# ``setup_experiment`` is sync and races on the next-index probe, so we
# serialise the directory-allocation step behind an asyncio.Lock; once a
# task owns its directory the actual game runs are unsynchronised.
#
# Output policy: we used to print "Starting game N" / "Game N done" plus
# whatever the engine spewed (per-turn [DEBUG] regex matches, player
# initialisation banners, etc.). For an 80-game sweep that ballooned
# the notebook to 50+ MB. Now: one tqdm bar per (model, config) pair
# shows completed/total games, and each game's stdout/stderr is
# redirected to ``{experiment_path}/game_{i}.log`` so the per-turn
# detail is still recoverable from disk if a game misbehaves.

import contextlib
from tqdm.auto import tqdm

_SETUP_LOCK = asyncio.Lock()

winner_map = {1: "Impostors (outnumber)", 2: "Crewmates (ejection)",
              3: "Crewmates (tasks)", 4: "Impostors (timeout)"}


async def run_games_for_config(
    model_key: str,
    config_name: str,
    cfg: dict,
    num_games: int,
    semaphore: asyncio.Semaphore,
    exp_path: str,
    pbar: "tqdm",
) -> list[dict]:
    """Run ``num_games`` Among Us games for one (model, config) pair.

    Assumes the calling task has already set ``MODEL_KEY_VAR`` and
    ``EXP_PATH_VAR``; child game tasks inherit them automatically.
    """
    async def run_single_game(game_index: int):
        async with semaphore:
            log_path = os.path.join(exp_path, f"game_{game_index}.log")
            # Redirect the engine's chatter (per-turn [DEBUG] regex
            # parses, "Initializing player ..." banners, etc.) to a
            # per-game logfile so it doesn't crowd out the tqdm bars.
            # Errors are still surfaced via the asyncio.gather below.
            with open(log_path, "w") as logf, \
                 contextlib.redirect_stdout(logf), \
                 contextlib.redirect_stderr(logf):
                game = AmongUs(
                    game_config=cfg,
                    include_human=False,
                    test=False,
                    personality=False,
                    agent_config=AGENT_CONFIG,
                    UI=None,
                    game_index=game_index,
                )
                game.collect_epistemic = True
                winner = await game.run_game()
                n_snap = len(game.epistemic_log)
            pbar.update(1)
            # Show the most recent winner inline so the bar is informative
            # at a glance even before the config finishes.
            pbar.set_postfix_str(
                f"last g{game_index}={winner_map.get(winner, str(winner))[:18]} "
                f"snap={n_snap}",
                refresh=False,
            )
            return {"game_index": game_index, "winner": winner,
                    "snapshots": n_snap,
                    "config": config_name, "model": model_key}

    tasks = [run_single_game(i) for i in range(1, num_games + 1)]
    return list(await asyncio.gather(*tasks))


async def run_sweep_for_config(
    model_key: str,
    cfg_name: str,
    cfg: dict,
    semaphore: asyncio.Semaphore,
) -> tuple[tuple[str, str], dict]:
    """Allocate this config's experiment directory and run its games."""
    cfg_game_args = dict(GAME_ARGS, game_config=cfg)
    async with _SETUP_LOCK:
        EXPERIMENT_NAME = setup_experiment(
            experiment_name=f"{model_key.replace('.', '_')}_{cfg_name}",
            LOGS_PATH=LOGS_PATH,
            DATE=DATE,
            COMMIT_HASH=COMMIT_HASH,
            DEFAULT_ARGS=cfg_game_args,
        )
        EXPERIMENT_PATH = os.environ["EXPERIMENT_PATH"]
    EXP_PATH_VAR.set(EXPERIMENT_PATH)

    # One bar per (model, config). ``tqdm.auto`` picks the widget
    # backend in a notebook and the text backend on a plain terminal.
    pbar = tqdm(
        total=NUM_GAMES_PER_CONFIG,
        desc=f"{model_key:<32s} {cfg_name}",
        unit="game",
        leave=True,
    )
    pbar.write(
        f"  [{model_key}|{cfg_name}] dir={EXPERIMENT_NAME}  "
        f"(crew={cfg['num_players']-cfg['num_impostors']}, "
        f"imp={cfg['num_impostors']})  "
        f"per-game logs in {EXPERIMENT_PATH}/game_*.log"
    )

    try:
        results = await run_games_for_config(
            model_key, cfg_name, cfg, NUM_GAMES_PER_CONFIG,
            semaphore=semaphore,
            exp_path=EXPERIMENT_PATH,
            pbar=pbar,
        )
    finally:
        pbar.close()

    wins = [winner_map.get(r["winner"], r["winner"]) for r in results]
    print(f"  [{model_key}|{cfg_name}] Outcomes: {wins}")

    return (model_key, cfg_name), {
        "experiment_name": EXPERIMENT_NAME,
        "experiment_path": EXPERIMENT_PATH,
        "config": cfg,
        "model": model_key,
        "games": results,
    }


async def run_sweep_for_model(model_key: str) -> dict:
    """Run all GAME_CONFIGS for a single model — configs in parallel.

    Concurrency: ``len(GAME_CONFIGS)`` config tasks each spawn
    ``NUM_GAMES_PER_CONFIG`` game tasks. They all share a single
    per-model semaphore so the backend never sees more than
    ``rate_limit`` concurrent in-flight requests for this model.
    """
    spec = MODEL_BACKENDS[model_key]
    model_rate = spec.get("rate_limit", RATE_LIMIT)
    # Bind backend in this task's context — inherited by every game/agent
    # spawned beneath it (configs and games alike).
    MODEL_KEY_VAR.set(model_key)
    semaphore = asyncio.Semaphore(model_rate)

    print(f"# MODEL: {model_key}  ({spec['backend']} | {spec['model_id']})  "
          f"rate_limit={model_rate} (shared across {len(GAME_CONFIGS)} configs)")

    cfg_tasks = [
        asyncio.create_task(
            run_sweep_for_config(model_key, cfg_name, cfg, semaphore),
            name=f"cfg:{model_key}:{cfg_name}",
        )
        for cfg_name, cfg in GAME_CONFIGS.items()
    ]
    cfg_pairs = await asyncio.gather(*cfg_tasks)
    return {key: val for key, val in cfg_pairs}


# ── Launch: one task per model, all run concurrently ────────────────
print(f"Launching {len(MODELS_TO_RUN)} parallel model sweeps: {MODELS_TO_RUN}")
print(f"  ({len(MODELS_TO_RUN)} models × {len(GAME_CONFIGS)} configs × "
      f"{NUM_GAMES_PER_CONFIG} games "
      f"= {len(MODELS_TO_RUN)*len(GAME_CONFIGS)*NUM_GAMES_PER_CONFIG} total)")
print("Per-game stdout/stderr is redirected to "
      "{experiment_path}/game_{i}.log; vLLM server logs are in "
      f"{VLLM_LOG_DIR if 'VLLM_LOG_DIR' in dir() else '<vllm-logs>'}.\n")

_t0 = time.time()
_model_tasks = [
    asyncio.create_task(run_sweep_for_model(m), name=f"sweep:{m}")
    for m in MODELS_TO_RUN
]
_model_outputs = await asyncio.gather(*_model_tasks)

sweep_results: "dict[tuple[str, str], dict]" = {}
for d in _model_outputs:
    sweep_results.update(d)

total_runs = len(sweep_results)
total_games = total_runs * NUM_GAMES_PER_CONFIG
print("\n" + "=" * 70)
print(f"SWEEP COMPLETE — {len(MODELS_TO_RUN)} models × "
      f"{len(GAME_CONFIGS)} configs × {NUM_GAMES_PER_CONFIG} games "
      f"= {total_games} total games  "
      f"(wall: {time.time() - _t0:.1f}s)")
print("=" * 70)

# ── Token usage & cost report (uses caching telemetry from cell 8) ──
# Pricing as of 2026-04 (per 1M tokens). Cached input is the discounted
# rate the providers charge for cache HITS. Both backends report cached
# tokens in their ``usage`` block, so this is exact billing — not a
# proxy estimate.
PRICING = {
    # OpenAI GPT-5.4 family (extrapolated from current GPT-5 tier ratios:
    # nano ≈ 1/25 of full, mini ≈ 1/5 of full).
    "gpt-5.4-nano":       {"in": 0.05,  "cached_in": 0.005, "out":  0.40},
    "gpt-5.4-mini":       {"in": 0.25,  "cached_in": 0.025, "out":  2.00},
    "gpt-5.4":            {"in": 1.25,  "cached_in": 0.125, "out": 10.00},
    # Anthropic Claude family.
    "claude-haiku-4-5":   {"in": 0.80,  "cached_in": 0.08,  "out":  4.00,
                           "cache_write": 1.00},
    "claude-sonnet-4-6":  {"in": 3.00,  "cached_in": 0.30,  "out": 15.00,
                           "cache_write": 3.75},
}

if TOKEN_USAGE:
    print("\n" + "=" * 70)
    print("TOKEN USAGE & ESTIMATED COST")
    print("=" * 70)
    grand = 0.0
    for m, u in TOKEN_USAGE.items():
        p = PRICING.get(m, {"in": 0, "cached_in": 0, "out": 0})
        fresh_in = max(0, u["in"] - u.get("cached_in", 0))
        cached_in = u.get("cached_in", 0)
        out_tok = u["out"] + u.get("reasoning", 0)  # GPT bills hidden reasoning at output rate
        cost = (fresh_in * p["in"] + cached_in * p.get("cached_in", p["in"])
                + out_tok * p["out"]) / 1e6
        grand += cost
        hit_rate = (cached_in / u["in"]) if u["in"] else 0.0
        print(f"  {m}")
        print(f"    calls       : {u['calls']:>8,}")
        print(f"    in (fresh)  : {fresh_in:>10,}  @ ${p['in']:.2f}/M")
        print(f"    in (cached) : {cached_in:>10,}  @ ${p.get('cached_in', p['in']):.3f}/M  "
              f"(hit rate: {hit_rate:.1%})")
        print(f"    out         : {u['out']:>10,}  @ ${p['out']:.2f}/M")
        if u.get("reasoning"):
            print(f"    reasoning   : {u['reasoning']:>10,}  (billed at output rate)")
        print(f"    -> ${cost:.3f}")
    print(f"\n  GRAND TOTAL : ${grand:.2f}")
else:
    print("\n(No paid-API token usage recorded — only local vLLM was used.)")


---
## 5. Compute Theory-of-Mind Metrics

Parse the `epistemic-states.jsonl` log, build ground truth from `summary.json`,
and evaluate all 8 metrics:

| # | Metric | Measures |
|---|--------|----------|
| 1 | Deceptive Efficacy | Impostor speech reducing suspicion |
| 2 | Detection Accuracy | Belief calibration vs ground truth |
| 3 | Social Influence | Speaker's effect on group voting |
| 4 | Faction Consensus | Within-faction voting agreement |
| 5 | Belief Volatility | Susceptibility to persuasion |
| 6 | Alibi Grounding | Claimed vs actual room visits |
| 7 | Adversarial Coordination | Zero-shot Impostor synergy |
| 8 | Objective Viability | Task completion x survival tradeoff |

In [ ]:
# ── Force-reload the metrics module so any edits to
# evaluations/metrics_calculator.py take effect without a kernel restart.
import importlib
import evaluations.metrics_calculator as _mc
_mc = importlib.reload(_mc)
process_experiment = _mc.process_experiment
EpistemicLogParser = _mc.EpistemicLogParser
TheoryOfMindEvaluator = _mc.TheoryOfMindEvaluator

# ── Compute metrics for every (model, config) experiment ─────
all_dfs: list[pd.DataFrame] = []

for (model_key, cfg_name), info in sweep_results.items():
    exp_path = info["experiment_path"]
    df_x = process_experiment(exp_path)
    if df_x is None or df_x.empty:
        print(f"[{model_key}|{cfg_name}] No epistemic data — skipping.")
        continue
    df_x = df_x.copy()
    df_x["config"] = cfg_name
    df_x["model"] = model_key

    out_csv = os.path.join(
        RESULTS_PATH, f"{info['experiment_name']}_tom_metrics.csv"
    )
    df_x.to_csv(out_csv, index=False)
    print(f"[{model_key}|{cfg_name}] {len(df_x)} rows -> {out_csv}")
    all_dfs.append(df_x)

if all_dfs:
    df_metrics = pd.concat(all_dfs, ignore_index=True)
    combined_csv = os.path.join(
        RESULTS_PATH, f"{DATE}_tom_metrics_sweep.csv"
    )
    df_metrics.to_csv(combined_csv, index=False)
    print(f"\nCombined: {len(df_metrics)} rows | "
          f"{df_metrics['model'].nunique()} models × "
          f"{df_metrics['config'].nunique()} configs  "
          f"-> {combined_csv}")
    print("\nRows per (model, config):")
    print(df_metrics.groupby(["model", "config"]).size().to_string())
else:
    print("No metrics collected.")
    df_metrics = pd.DataFrame()

---
## 6. Results

### 6a. Full Per-Player Metrics Table

In [ ]:
if not df_metrics.empty:
    df_display = df_metrics.copy()
    df_display["role"] = df_display["identity"].map({0: "Crewmate", 1: "Impostor"})

    metric_cols = [
        "detection_accuracy", "social_influence", "social_influence_signed",
        "belief_volatility", "belief_volatility_signed",
        "alibi_grounding", "alibi_grounding_graph", "alibi_consistency",
        "objective_viability", "faction_consensus",
        "deceptive_efficacy", "spatial_dispersion", "alibi_corroboration",
    ]
    display_cols = ["player", "role", "timestep"] + [
        c for c in metric_cols if c in df_display.columns
    ]

    styled = (
        df_display[display_cols]
        .style
        .format({c: "{:.4f}" for c in metric_cols if c in df_display.columns})
        .set_caption("Per-Player Theory-of-Mind Metrics")
    )
    display(styled)
else:
    print("No metrics to display.")

### 6b. Role-Segregated Metrics

Metrics split by which role they're actually designed for:

| Metric | Impostor | Crewmate | Notes |
|--------|:---:|:---:|-------|
| **Deceptive Efficacy** | ✅ primary | — | How well impostors reduce suspicion on themselves |
| **Detection Accuracy** | — | ✅ primary | How well crewmates identify the impostor (lower is better) |
| **Spatial Dispersion** | ✅ stealth spread | ✅ task coverage | Avg shortest-path between same-faction players |
| **Alibi Corroboration** | ✅ collusion signal | ✅ mutual witness | Jaccard of claimed regions within faction |
| **Alibi Grounding** | ⚠️ low = lying | ⚠️ high = honest | Jaccard(claimed rooms, actual rooms) |
| **Social Influence** | ✅ both | ✅ both | KL shift of group vote after speech |
| **Belief Volatility** | ✅ both | ✅ both | Susceptibility to others' speech |
| **Faction Consensus** | ✅ per-faction | ✅ per-faction | Within-faction voting entropy |
| **Objective Viability** | ✅ role-aware | ✅ role-aware | Harmonic mean of *productivity × survival*. **Crewmate**: productivity = tasks-completed / tasks-assigned. **Impostor**: productivity = kills / total-crewmates. Both factions share the survival term. |

### 6c. Zero-Shot Coordination Over Time (by faction)

Evolution of faction-level coordination signals (spatial dispersion and alibi corroboration) across meeting timesteps, compared for Impostors vs. Crewmates.

In [ ]:
if not df_metrics.empty:
    df_c = df_metrics.copy()
    df_c["role"] = df_c["identity"].map({0: "Crewmate", 1: "Impostor"})
    coord_cols = [c for c in ["spatial_dispersion", "alibi_corroboration"]
                  if c in df_c.columns]

    if coord_cols:
        # Faction x Timestep pivot, mean of each coordination metric
        grouped = (
            df_c.groupby(["role", "timestep"])[coord_cols]
            .mean()
            .round(4)
        )
        print("ZERO-SHOT COORDINATION — BY FACTION × MEETING TIMESTEP")
        print("-" * 72)
        display(
            grouped.style
            .format("{:.4f}")
            .set_caption("Spatial dispersion & alibi corroboration — both factions")
        )

        # Compact per-role aggregate
        agg = (
            df_c.groupby("role")[coord_cols]
            .agg(["mean", "std"])
            .round(4)
        )
        agg.columns = [f"{c}_{s}" for c, s in agg.columns]
        print("\nFaction aggregate (across all meetings):")
        display(agg.style.format("{:.4f}"))

        print("\nInterpretation:")
        print("  spatial_dispersion  (Impostor)  -> higher = spread out, harder to jointly catch")
        print("  spatial_dispersion  (Crewmate)  -> higher = better task-map coverage")
        print("  alibi_corroboration (Impostor)  -> higher = synchronised claims, collusion signal")
        print("  alibi_corroboration (Crewmate)  -> higher = mutual witnessing / overlapping routes")
    else:
        print("No coordination metrics present in df_metrics.")
else:
    print("No metrics to display.")

### 6d. Descriptive Statistics — Pooled (all roles)

Overall distribution of each metric across every player and meeting (both factions combined). For role-specific quartile tables see section 6b.

In [ ]:
if not df_metrics.empty:
    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ["identity", "timestep"]]

    desc = df_metrics[numeric_cols].describe().T
    desc = desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]

    print("DESCRIPTIVE STATISTICS — ALL METRICS")
    print("=" * 80)
    display(
        desc.style
        .format("{:.4f}")
        .set_caption("Descriptive Statistics for All Theory-of-Mind Metrics")
    )
else:
    print("No metrics to describe.")

### 6e. Game Example — Meeting Replay with Epistemic States

Pick a game and walk through the meeting: who said what, what each agent
privately believed, and how those beliefs shifted after the discussion.

In [ ]:
# ── Game Example: pick the first game that has epistemic data ─────
import textwrap
from IPython.display import display, HTML

# Choose which config's experiment to preview. Defaults to the first
# configuration with non-empty epistemic data.
EXAMPLE_CONFIG = None  # e.g. "5C_2I"; None = auto-pick first non-empty
_example_path = None
if "sweep_results" in globals() and sweep_results:
    candidates = (
        [EXAMPLE_CONFIG] if EXAMPLE_CONFIG else list(sweep_results.keys())
    )
    for _cname in candidates:
        _p = sweep_results.get(_cname, {}).get("experiment_path")
        if _p and os.path.exists(os.path.join(_p, "epistemic-states.jsonl")):
            _example_path = _p
            EXAMPLE_CONFIG = _cname
            break
EXPERIMENT_PATH = _example_path or globals().get("EXPERIMENT_PATH", "")
print(f"Previewing config: {EXAMPLE_CONFIG} — {EXPERIMENT_PATH}")

summary_path = os.path.join(EXPERIMENT_PATH, "summary.json")
logs_path = os.path.join(EXPERIMENT_PATH, "agent-logs-compact.json")
epistemic_path = os.path.join(EXPERIMENT_PATH, "epistemic-states.jsonl")

# ── 1. Load game summaries and pick one with a meeting ────────────
summaries = []
with open(summary_path) as f:
    for line in f:
        line = line.strip()
        if line:
            summaries.append(json.loads(line))

# Load all epistemic snapshots
epi_snaps = []
with open(epistemic_path) as f:
    for line in f:
        line = line.strip()
        if line:
            epi_snaps.append(json.loads(line))

# Find which game indices have epistemic data
games_with_epi = set()
for s in epi_snaps:
    meta = s.get("_meta", {})
    # game index is embedded in the experiment — match via timestep presence
    games_with_epi.add(meta.get("timestep", -1))

# Pick the first summary that has meeting data
chosen_summary = None
chosen_game_key = None
for summary in summaries:
    for key in summary:
        if key.startswith("Game"):
            chosen_summary = summary
            chosen_game_key = key
            break
    if chosen_summary:
        break

if not chosen_summary:
    print("No game data found in summary.")
else:
    game_data = chosen_summary[chosen_game_key]
    game_idx_str = chosen_game_key  # e.g. "Game 4"
    winner_reason = game_data.get("winner_reason", "Unknown")

    # ── 2. Build player roster ────────────────────────────────────
    roster = {}
    for pkey, pval in game_data.items():
        if pkey.startswith("Player ") and isinstance(pval, dict):
            name = pval["name"]
            roster[name] = {
                "identity": pval["identity"],
                "color": pval["color"],
                "tasks": pval.get("tasks", []),
            }

    impostor_names = [n for n, r in roster.items() if r["identity"] == "Impostor"]

    # ── 3. Extract meeting speech from compact logs ───────────────
    meeting_speech = []  # (game_index, step, player, speech_text)
    with open(logs_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            if entry.get("game_index") != game_idx_str:
                continue
            pname = entry.get("player", {}).get("name", "")
            full = entry.get("interaction", {}).get("full_response", "")
            step = entry.get("step", "?")
            if not isinstance(full, str):
                continue

            # Extract SPEAK content
            speak_match = re.search(r'SPEAK:\s*"(.*?)"', full, re.DOTALL)
            if not speak_match:
                speak_match = re.search(r'SPEAK:\s*(.*?)$', full, re.DOTALL | re.MULTILINE)
            if speak_match:
                speech = speak_match.group(1).strip().strip('"')
                meeting_speech.append((step, pname, speech))

            # Extract VOTE
            vote_match = re.search(r'\[?Action\]?\s*VOTE\s+(.*)', full, re.IGNORECASE)
            if vote_match:
                vote_target = vote_match.group(1).strip()
                meeting_speech.append((step, pname, f"[VOTE] {vote_target}"))

    # ── 4. Get epistemic snapshots for this game ──────────────────
    game_epi = [s for s in epi_snaps
                if any(s.get("_meta", {}).get("player", "") in roster for _ in [1])]

    pre_snaps = [s for s in game_epi if s.get("_meta", {}).get("phase_label") == "pre_meeting"]
    post_snaps = [s for s in game_epi if s.get("_meta", {}).get("phase_label") == "post_meeting"]

    # Group by timestep
    pre_by_t = {}
    for s in pre_snaps:
        t = s["_meta"]["timestep"]
        pre_by_t.setdefault(t, []).append(s)
    post_by_t = {}
    for s in post_snaps:
        t = s["_meta"]["timestep"]
        post_by_t.setdefault(t, []).append(s)

    # ── 5. Render ─────────────────────────────────────────────────
    role_icon = lambda name: "🔪" if name in impostor_names else "👤"
    role_tag = lambda name: "IMPOSTOR" if name in impostor_names else "Crewmate"

    html_parts = []
    html_parts.append(f"<h3>{game_idx_str} — {winner_reason}</h3>")

    # Roster table
    html_parts.append("<h4>Player Roster</h4>")
    html_parts.append("<table style='border-collapse:collapse; margin-bottom:16px;'>")
    html_parts.append("<tr><th style='padding:4px 12px; border:1px solid #ccc;'>Player</th>"
                       "<th style='padding:4px 12px; border:1px solid #ccc;'>Role</th>"
                       "<th style='padding:4px 12px; border:1px solid #ccc;'>Tasks</th></tr>")
    for name, info in roster.items():
        role_style = "color:red; font-weight:bold;" if info["identity"] == "Impostor" else ""
        html_parts.append(
            f"<tr><td style='padding:4px 12px; border:1px solid #ccc;'>{role_icon(name)} {name}</td>"
            f"<td style='padding:4px 12px; border:1px solid #ccc; {role_style}'>{info['identity']}</td>"
            f"<td style='padding:4px 12px; border:1px solid #ccc;'>{', '.join(info['tasks'])}</td></tr>"
        )
    html_parts.append("</table>")

    # Meeting speech timeline
    if meeting_speech:
        html_parts.append("<h4>Meeting Discussion</h4>")
        current_step = None
        for step, pname, speech in meeting_speech:
            if step != current_step:
                current_step = step
                html_parts.append(f"<div style='margin-top:12px; padding:4px 8px; "
                                   f"background:#e8e8e8; font-weight:bold; border-radius:4px;'>"
                                   f"Timestep {step}</div>")
            is_imp = pname in impostor_names
            bg = "#ffe0e0" if is_imp else "#e0f0e0"
            icon = role_icon(pname)
            short_speech = speech[:300] + ("..." if len(speech) > 300 else "")
            html_parts.append(
                f"<div style='margin:4px 0 4px 16px; padding:6px 10px; "
                f"background:{bg}; border-radius:6px; border-left:4px solid {'#c00' if is_imp else '#0a0'};'>"
                f"<b>{icon} {pname}</b>: {short_speech}</div>"
            )

    # Epistemic state snapshots
    all_timesteps = sorted(set(list(pre_by_t.keys()) + list(post_by_t.keys())))
    if all_timesteps:
        html_parts.append("<h4>Epistemic State Snapshots</h4>")
        for t in all_timesteps:
            html_parts.append(f"<div style='margin-top:14px; padding:4px 8px; "
                               f"background:#d0d8f0; font-weight:bold; border-radius:4px;'>"
                               f"Timestep {t}</div>")

            for phase, snaps_dict in [("PRE-meeting", pre_by_t), ("POST-meeting", post_by_t)]:
                snaps = snaps_dict.get(t, [])
                if not snaps:
                    continue
                html_parts.append(f"<div style='margin:6px 0 2px 8px; font-weight:bold; "
                                   f"color:#555;'>{phase}</div>")

                # Build a beliefs table
                html_parts.append(
                    "<table style='border-collapse:collapse; margin:4px 0 8px 16px; font-size:13px;'>"
                )
                # Header: player names as columns
                all_targets = set()
                for s in snaps:
                    all_targets.update(s.get("belief_distribution", {}).keys())
                    all_targets.update(k for k in s.get("voting_intent", {}).keys() if k != "Skip")
                target_names = sorted(all_targets)

                html_parts.append("<tr><th style='padding:3px 8px; border:1px solid #ccc;'>Agent</th>")
                for tn in target_names:
                    short = tn.split(":")[0] if ":" in tn else tn
                    html_parts.append(f"<th style='padding:3px 6px; border:1px solid #ccc;'>{short}<br>"
                                       f"<span style='font-size:10px;'>belief / vote</span></th>")
                html_parts.append("<th style='padding:3px 6px; border:1px solid #ccc;'>Skip<br>"
                                   "<span style='font-size:10px;'>vote</span></th>")
                html_parts.append("<th style='padding:3px 8px; border:1px solid #ccc; max-width:300px;'>"
                                   "Reasoning</th></tr>")

                for s in snaps:
                    meta = s.get("_meta", {})
                    agent_name = meta.get("player", "?")
                    beliefs = s.get("belief_distribution", {})
                    votes = s.get("voting_intent", {})
                    reasoning = s.get("reasoning_scratchpad", "")
                    is_imp = agent_name in impostor_names
                    row_bg = "#fff0f0" if is_imp else "#f8fff8"

                    html_parts.append(f"<tr style='background:{row_bg};'>")
                    html_parts.append(f"<td style='padding:3px 8px; border:1px solid #ccc; "
                                       f"font-weight:bold;'>{role_icon(agent_name)} "
                                       f"{agent_name.split(':')[0]}</td>")
                    for tn in target_names:
                        b = beliefs.get(tn, "")
                        v = votes.get(tn, "")
                        b_str = f"{b:.2f}" if isinstance(b, (int, float)) else "-"
                        v_str = f"{v:.2f}" if isinstance(v, (int, float)) else "-"
                        # Highlight high suspicion
                        b_color = ""
                        if isinstance(b, (int, float)) and b >= 0.7:
                            b_color = "color:red; font-weight:bold;"
                        html_parts.append(
                            f"<td style='padding:3px 6px; border:1px solid #ccc; text-align:center;'>"
                            f"<span style='{b_color}'>{b_str}</span> / {v_str}</td>"
                        )
                    skip_v = votes.get("Skip", "")
                    skip_str = f"{skip_v:.2f}" if isinstance(skip_v, (int, float)) else "-"
                    html_parts.append(f"<td style='padding:3px 6px; border:1px solid #ccc; "
                                       f"text-align:center;'>{skip_str}</td>")
                    short_reason = reasoning[:120] + ("..." if len(reasoning) > 120 else "")
                    html_parts.append(f"<td style='padding:3px 8px; border:1px solid #ccc; "
                                       f"font-size:11px; max-width:300px;'>{short_reason}</td>")
                    html_parts.append("</tr>")

                html_parts.append("</table>")

    # Metrics for this game (from df_metrics)
    if not df_metrics.empty:
        game_metrics = df_metrics[df_metrics["experiment"] == os.path.basename(EXPERIMENT_PATH)]
        if not game_metrics.empty:
            first_t = all_timesteps[0] if all_timesteps else None
            if first_t is not None:
                t_metrics = game_metrics[game_metrics["timestep"] == first_t]
            else:
                t_metrics = game_metrics.head(5)

            if not t_metrics.empty:
                html_parts.append("<h4>Metrics Snapshot (first meeting)</h4>")
                float_cols = [c for c in t_metrics.columns
                              if t_metrics[c].dtype in ["float64", "float32"]]
                fmt = {c: "{:.4f}" for c in float_cols}
                html_parts.append(
                    t_metrics[["player", "timestep"] + float_cols]
                    .to_html(index=False, float_format="%.4f",
                             classes="table", border=1)
                )

    display(HTML("\n".join(html_parts)))

### 6f. Cross-Configuration Comparison (mean ± std)

Two tables — one per faction — summarising every metric across the four game configurations `4C_1I`, `4C_2I`, `5C_1I`, `5C_2I`.

- **Rows** = metric names
- **Columns** = configurations (optionally grouped by model when multiple models are run)
- **Cells** = `mean ± std` across all *(game × meeting × player-of-that-faction)* rows for that (model, config) pair
- **`n (rows)`** footer shows how many observations contribute to each column.

When `MODELS_TO_RUN` contains more than one model, columns use a two-level index `(model, config)` so you can read Qwen vs GPT vs Claude side-by-side for the same configuration.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cross-config comparison tables — one per faction.
# Rows = metrics, columns = configurations, cells = "mean ± std".
# ══════════════════════════════════════════════════════════════════

if df_metrics.empty or "config" not in df_metrics.columns:
    print("df_metrics is empty or missing 'config' column — nothing to compare.")
else:
    IMPOSTOR_METRICS = [
        "deceptive_efficacy",
        "spatial_dispersion",
        "alibi_corroboration",
        "alibi_grounding",
        "social_influence",
        "belief_volatility",
        "faction_consensus",
        "objective_viability",
    ]
    CREWMATE_METRICS = [
        "detection_accuracy",
        "spatial_dispersion",
        "alibi_corroboration",
        "alibi_grounding",
        "social_influence",
        "belief_volatility",
        "faction_consensus",
        "objective_viability",
    ]

    df_c = df_metrics.copy()
    df_c["role"] = df_c["identity"].map({0: "Crewmate", 1: "Impostor"})
    if "model" not in df_c.columns:
        df_c["model"] = "qwen3-4b"

    config_order = [c for c in GAME_CONFIGS if c in df_c["config"].unique()]
    model_order = [m for m in MODELS_TO_RUN if m in df_c["model"].unique()]
    if not model_order:
        model_order = sorted(df_c["model"].unique().tolist())

    def _cell(vals: pd.Series) -> str:
        vals = vals.dropna()
        if len(vals) == 0:
            return "—"
        if len(vals) == 1:
            return f"{vals.iloc[0]:.4f}"
        return f"{vals.mean():.4f} ± {vals.std():.4f}"

    def build_meanstd_table(
        sub: pd.DataFrame, metrics: list[str],
        models: list[str], configs: list[str],
    ) -> pd.DataFrame:
        """Rows = metrics, columns = MultiIndex(model, config).

        Each cell is 'mean ± std' across all matching rows of ``sub``.
        Falls back to a flat (config-only) index if only one model is
        present, for readability.
        """
        col_tuples = [(m, c) for m in models for c in configs]
        cell_map: dict[tuple[str, str], dict[str, str]] = {
            col: {} for col in col_tuples
        }
        ns: dict[tuple[str, str], int] = {}
        for m_key in models:
            for cfg_name in configs:
                s = sub[(sub["model"] == m_key) & (sub["config"] == cfg_name)]
                ns[(m_key, cfg_name)] = len(s)
                for met in metrics:
                    if met in s.columns:
                        cell_map[(m_key, cfg_name)][met] = _cell(s[met])
                    else:
                        cell_map[(m_key, cfg_name)][met] = "—"

        data = {col: [cell_map[col][met] for met in metrics]
                for col in col_tuples}
        out = pd.DataFrame(data, index=metrics)
        out.columns = pd.MultiIndex.from_tuples(
            col_tuples, names=["model", "config"]
        )
        out.index.name = "metric"
        out.loc["n (rows)"] = [str(ns[col]) for col in col_tuples]
        if len(models) == 1:
            out.columns = [c for _, c in out.columns]
        return out

    # ── IMPOSTOR TABLE ────────────────────────────────────────────
    imp_sub = df_c[df_c["role"] == "Impostor"]
    print("=" * 80)
    print("IMPOSTOR — metrics × (model, configuration)  (mean ± std)")
    print("=" * 80)
    if imp_sub.empty:
        print("  No Impostor rows across any (model, configuration).")
    else:
        imp_table = build_meanstd_table(
            imp_sub, IMPOSTOR_METRICS, model_order, config_order
        )
        display(
            imp_table.style
            .set_caption("Impostor metrics across (model × configuration)")
            .set_properties(**{"text-align": "center"})
        )

    # ── CREWMATE TABLE ────────────────────────────────────────────
    crew_sub = df_c[df_c["role"] == "Crewmate"]
    print("\n" + "=" * 80)
    print("CREWMATE — metrics × (model, configuration)  (mean ± std)")
    print("=" * 80)
    if crew_sub.empty:
        print("  No Crewmate rows across any (model, configuration).")
    else:
        crew_table = build_meanstd_table(
            crew_sub, CREWMATE_METRICS, model_order, config_order
        )
        display(
            crew_table.style
            .set_caption("Crewmate metrics across (model × configuration)")
            .set_properties(**{"text-align": "center"})
        )

    imp_csv = os.path.join(RESULTS_PATH, f"{DATE}_impostor_x_model_x_config.csv")
    crew_csv = os.path.join(RESULTS_PATH, f"{DATE}_crewmate_x_model_x_config.csv")
    if not imp_sub.empty:
        imp_table.to_csv(imp_csv)
        print(f"\n  Impostor table saved -> {imp_csv}")
    if not crew_sub.empty:
        crew_table.to_csv(crew_csv)
        print(f"  Crewmate table saved -> {crew_csv}")

### 6g. Cross-Model Comparison (mean ± std)

Collapsing across all four configurations, how do models compare head-to-head?

- **Rows** = metrics
- **Columns** = models (`qwen3-4b`, `gpt-4o-mini`, `claude-3-5-haiku`, ...)
- **Cells** = `mean ± std` pooled across every (config × game × meeting × player-of-faction) row
- Also computes a `rank_sum` row — for each model, the sum of its ranks across metrics (lower is "better", but interpret cautiously: some metrics are *higher-better* for one faction and *lower-better* for the other).

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cross-model comparison — pool across all configurations.
# One table per faction: rows = metrics, columns = models, cells = mean ± std.
# ══════════════════════════════════════════════════════════════════

if df_metrics.empty or "model" not in df_metrics.columns:
    print("df_metrics has no 'model' column — run the multi-model sweep first.")
else:
    df_m = df_metrics.copy()
    df_m["role"] = df_m["identity"].map({0: "Crewmate", 1: "Impostor"})

    model_order = [m for m in MODELS_TO_RUN if m in df_m["model"].unique()]
    if not model_order:
        model_order = sorted(df_m["model"].unique().tolist())

    def _cellv(vals: pd.Series) -> str:
        vals = vals.dropna()
        if len(vals) == 0:
            return "—"
        if len(vals) == 1:
            return f"{vals.iloc[0]:.4f}"
        return f"{vals.mean():.4f} ± {vals.std():.4f}"

    def build_model_table(
        sub: pd.DataFrame, metrics: list[str], models: list[str]
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        """Return (formatted_table, numeric_means) for ranking."""
        fmt: dict[str, list[str]] = {m: [] for m in models}
        num: dict[str, list[float]] = {m: [] for m in models}
        ns: dict[str, int] = {}
        for m_key in models:
            s = sub[sub["model"] == m_key]
            ns[m_key] = len(s)
            for met in metrics:
                if met not in s.columns:
                    fmt[m_key].append("—")
                    num[m_key].append(float("nan"))
                    continue
                fmt[m_key].append(_cellv(s[met]))
                num[m_key].append(float(s[met].mean()) if s[met].notna().any()
                                  else float("nan"))
        tbl = pd.DataFrame(fmt, index=metrics)
        tbl.index.name = "metric"
        tbl.loc["n (rows)"] = [str(ns[m]) for m in models]
        means = pd.DataFrame(num, index=metrics)
        return tbl, means

    # ── IMPOSTOR ──────────────────────────────────────────────────
    imp_sub = df_m[df_m["role"] == "Impostor"]
    print("=" * 80)
    print("IMPOSTOR — metrics × models  (pooled across configs, mean ± std)")
    print("=" * 80)
    if imp_sub.empty:
        print("  No Impostor rows.")
    else:
        imp_tbl, imp_means = build_model_table(
            imp_sub, IMPOSTOR_METRICS, model_order
        )
        display(
            imp_tbl.style
            .set_caption("Impostor — cross-model comparison")
            .set_properties(**{"text-align": "center"})
        )
        # Simple per-metric rank (1 = highest mean). Interpret direction
        # per metric; e.g. deceptive_efficacy higher is better.
        imp_rank = imp_means.rank(axis=1, ascending=False, method="average")
        imp_rank.loc["rank_sum"] = imp_rank.sum(axis=0, skipna=True)
        print("\n  Impostor — per-metric rank (1 = highest-mean model):")
        display(imp_rank.round(2))

    # ── CREWMATE ──────────────────────────────────────────────────
    crew_sub = df_m[df_m["role"] == "Crewmate"]
    print("\n" + "=" * 80)
    print("CREWMATE — metrics × models  (pooled across configs, mean ± std)")
    print("=" * 80)
    if crew_sub.empty:
        print("  No Crewmate rows.")
    else:
        crew_tbl, crew_means = build_model_table(
            crew_sub, CREWMATE_METRICS, model_order
        )
        display(
            crew_tbl.style
            .set_caption("Crewmate — cross-model comparison")
            .set_properties(**{"text-align": "center"})
        )
        # For crewmates, detection_accuracy is lower-is-better, so its
        # ranking should be flipped. Other metrics default to higher-better.
        crew_rank = crew_means.rank(axis=1, ascending=False, method="average")
        if "detection_accuracy" in crew_rank.index:
            crew_rank.loc["detection_accuracy"] = crew_means.loc[
                "detection_accuracy"
            ].rank(ascending=True, method="average")
        crew_rank.loc["rank_sum"] = crew_rank.sum(axis=0, skipna=True)
        print("\n  Crewmate — per-metric rank (lower rank = better; "
              "detection_accuracy uses lower-is-better):")
        display(crew_rank.round(2))

    # ── Export ────────────────────────────────────────────────────
    imp_csv = os.path.join(RESULTS_PATH, f"{DATE}_impostor_x_model.csv")
    crew_csv = os.path.join(RESULTS_PATH, f"{DATE}_crewmate_x_model.csv")
    if not imp_sub.empty:
        imp_tbl.to_csv(imp_csv)
        print(f"\n  Impostor × model table saved -> {imp_csv}")
    if not crew_sub.empty:
        crew_tbl.to_csv(crew_csv)
        print(f"  Crewmate × model table saved -> {crew_csv}")

### 6h. Win Rates by Model × Configuration (with Win-Category Breakdown)

Outcomes of every completed game, aggregated three ways.

**Winner codes (from `AmongUs.run_game`):**

| Code | Faction | Category |
|------|---------|----------|
| 1 | Impostors | **Outnumber** — crewmates eliminated / tied |
| 2 | Crewmates | **Ejection** — all impostors voted out |
| 3 | Crewmates | **Tasks** — all tasks completed |
| 4 | Impostors | **Timeout** — `max_timesteps` reached |

**Tables produced:**

1. **Faction win rate** by `(model, config)` — Crewmate vs Impostor share.
2. **Win-category breakdown** by `(model, config)` — share of games in each of the 4 categories.
3. **Marginal win rates by model** (pooled across configs) and **by config** (pooled across models).

Outcomes are loaded from in-memory `sweep_results` when available, and otherwise reconstructed by streaming each experiment's `summary.json` — so this cell works in a fresh kernel after a notebook restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Win-rate analysis: faction win rates and win-category breakdown
# across (model, config) combinations.
# ══════════════════════════════════════════════════════════════════

import matplotlib

WINNER_CATEGORY = {
    1: ("Impostor", "Outnumber"),
    2: ("Crewmate", "Ejection"),
    3: ("Crewmate", "Tasks"),
    4: ("Impostor", "Timeout"),
}
CATEGORY_ORDER = ["Ejection", "Tasks", "Outnumber", "Timeout"]


def _collect_game_outcomes() -> pd.DataFrame:
    """Build a long DataFrame: one row per finished game.

    Prefers in-memory ``sweep_results`` (populated by the sweep loop) and
    falls back to scanning each experiment's ``summary.json`` so the cell
    works after a kernel restart.
    """
    rows: list[dict] = []

    sw = globals().get("sweep_results", {}) or {}
    for (model_key, cfg_name), info in sw.items():
        for g in info.get("games", []) or []:
            w = g.get("winner")
            faction, category = WINNER_CATEGORY.get(w, ("Unknown", "Unknown"))
            rows.append({
                "model": model_key,
                "config": cfg_name,
                "game_index": g.get("game_index"),
                "winner_code": w,
                "winner_faction": faction,
                "winner_category": category,
                "source": "sweep_results",
            })

    if rows:
        return pd.DataFrame(rows)

    # ── Fallback: reconstruct from summary.json files ──────────────
    # ``experiment-details.txt`` cannot be trusted for the model identity:
    # it always records the static AGENT_CONFIG (Qwen path) because the
    # runtime model swap happens in-memory via configure_agent_for_model().
    # We recover the true (experiment_dir → model) mapping from the
    # already-persisted ToM-metrics sweep CSV (cell 13), which carries
    # both the ``experiment`` and ``model`` columns.
    print("sweep_results is empty — recovering (model, config) from "
          "tom_metrics_sweep CSV + summary.json files in LOGS_PATH.")

    import glob
    sweep_csvs = sorted(glob.glob(
        os.path.join(RESULTS_PATH, "*_tom_metrics_sweep.csv")
    ))
    exp_to_model: dict[str, str] = {}
    if sweep_csvs:
        latest_csv = sweep_csvs[-1]
        try:
            tom_df = pd.read_csv(latest_csv,
                                 usecols=["experiment", "model"])
            exp_to_model = (
                tom_df.dropna(subset=["experiment", "model"])
                .drop_duplicates(subset=["experiment"])
                .set_index("experiment")["model"]
                .to_dict()
            )
            print(f"  Loaded model labels for {len(exp_to_model)} "
                  f"experiments from {os.path.basename(latest_csv)}")
        except Exception as e:
            print(f"  Could not parse {latest_csv}: {e}")
    else:
        print(f"  No *_tom_metrics_sweep.csv found in {RESULTS_PATH} — "
              "model column will be marked 'unknown'.")

    for entry in sorted(os.listdir(LOGS_PATH)):
        exp_dir = os.path.join(LOGS_PATH, entry)
        details = os.path.join(exp_dir, "experiment-details.txt")
        summary = os.path.join(exp_dir, "summary.json")
        if not (os.path.isfile(details) and os.path.isfile(summary)):
            continue

        # Skip experiments that aren't part of any recorded sweep.
        if exp_to_model and entry not in exp_to_model:
            continue

        model_key = exp_to_model.get(entry, "unknown")

        # Game config (num_players / num_impostors) IS reliable in
        # experiment-details.txt — only the model field is misleading.
        cfg_name = "unknown"
        try:
            with open(details) as f:
                txt = f.read()
            cm = re.search(
                r"num_players'?:?\s*(\d+).*?num_impostors'?:?\s*(\d+)",
                txt, re.S,
            )
            if cm:
                n_p, n_i = int(cm.group(1)), int(cm.group(2))
                cfg_name = f"{n_p - n_i}C_{n_i}I"
        except Exception:
            pass

        with open(summary) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                for k, v in obj.items():
                    if not k.startswith("Game") or not isinstance(v, dict):
                        continue
                    w = v.get("winner")
                    faction, category = WINNER_CATEGORY.get(
                        w, ("Unknown", "Unknown")
                    )
                    rows.append({
                        "model": model_key,
                        "config": cfg_name,
                        "game_index": int(k.split()[-1]),
                        "winner_code": w,
                        "winner_faction": faction,
                        "winner_category": category,
                        "source": entry,
                    })

    return pd.DataFrame(rows)


df_wins = _collect_game_outcomes()

if df_wins.empty:
    print("No completed games found — run the sweep first.")
else:
    print(f"Loaded {len(df_wins)} games "
          f"({df_wins['model'].nunique()} models × "
          f"{df_wins['config'].nunique()} configs).")

    # Preserve canonical ordering when available.
    model_order = [m for m in globals().get("MODELS_TO_RUN", [])
                   if m in df_wins["model"].unique()]
    if not model_order:
        model_order = sorted(df_wins["model"].unique().tolist())
    config_order = [c for c in globals().get("GAME_CONFIGS", {})
                    if c in df_wins["config"].unique()]
    if not config_order:
        config_order = sorted(df_wins["config"].unique().tolist())

    df_wins["model"] = pd.Categorical(df_wins["model"],
                                      categories=model_order, ordered=True)
    df_wins["config"] = pd.Categorical(df_wins["config"],
                                       categories=config_order, ordered=True)

    # ── Table 1: faction win rate by (model, config) ──────────────
    grp = df_wins.groupby(["model", "config"], observed=True)
    n_games = grp.size().rename("n_games")
    crew_rate = (
        grp["winner_faction"].apply(lambda s: (s == "Crewmate").mean())
        .rename("crewmate_win_rate")
    )
    imp_rate = (
        grp["winner_faction"].apply(lambda s: (s == "Impostor").mean())
        .rename("impostor_win_rate")
    )
    faction_tbl = pd.concat([n_games, crew_rate, imp_rate], axis=1).reset_index()

    print("\n" + "=" * 80)
    print("FACTION WIN RATE — by (model, config)")
    print("=" * 80)
    display(
        faction_tbl.style
        .format({"crewmate_win_rate": "{:.2%}",
                 "impostor_win_rate": "{:.2%}",
                 "n_games": "{:d}"})
        .set_caption("Crewmate / Impostor share of wins per (model × config)")
    )

    # Also a wide pivoted view: rows=model, columns=config, cells=crewmate%
    crew_pivot = (
        faction_tbl.pivot(index="model", columns="config",
                          values="crewmate_win_rate")
        .reindex(index=model_order, columns=config_order)
    )
    imp_pivot = (
        faction_tbl.pivot(index="model", columns="config",
                          values="impostor_win_rate")
        .reindex(index=model_order, columns=config_order)
    )
    print("\nCrewmate win rate — rows: model, columns: config")
    display(crew_pivot.style.format("{:.2%}").background_gradient(
        cmap="Greens", vmin=0, vmax=1))
    print("Impostor win rate — rows: model, columns: config")
    display(imp_pivot.style.format("{:.2%}").background_gradient(
        cmap="Reds", vmin=0, vmax=1))

    # ── Table 2: win-category breakdown by (model, config) ────────
    cat_counts = (
        df_wins.groupby(["model", "config", "winner_category"],
                        observed=True).size()
        .unstack("winner_category", fill_value=0)
        .reindex(columns=CATEGORY_ORDER, fill_value=0)
    )
    cat_share = cat_counts.div(cat_counts.sum(axis=1), axis=0).fillna(0.0)

    print("\n" + "=" * 80)
    print("WIN-CATEGORY BREAKDOWN — share of games per category, by (model, config)")
    print("=" * 80)
    cat_share_disp = cat_share.copy()
    cat_share_disp.columns = pd.MultiIndex.from_tuples(
        [(("Crewmate" if c in ("Ejection", "Tasks") else "Impostor"), c)
         for c in cat_share_disp.columns],
        names=["faction", "category"],
    )
    display(
        cat_share_disp.style
        .format("{:.2%}")
        .background_gradient(cmap="Blues", vmin=0, vmax=1, axis=None)
        .set_caption("Fraction of games ending in each win category")
    )

    # Raw counts variant for transparency
    print("Raw game counts per category:")
    display(cat_counts.assign(total=cat_counts.sum(axis=1)))

    # ── Table 3: marginal win rates ────────────────────────────────
    def _faction_marginal(by: str) -> pd.DataFrame:
        m = df_wins.groupby(by, observed=True)
        out = pd.DataFrame({
            "n_games": m.size(),
            "crewmate_win_rate":
                m["winner_faction"].apply(lambda s: (s == "Crewmate").mean()),
            "impostor_win_rate":
                m["winner_faction"].apply(lambda s: (s == "Impostor").mean()),
        })
        cat = (
            df_wins.groupby([by, "winner_category"], observed=True).size()
            .unstack("winner_category", fill_value=0)
            .reindex(columns=CATEGORY_ORDER, fill_value=0)
        )
        cat_pct = cat.div(cat.sum(axis=1), axis=0).fillna(0.0)
        cat_pct.columns = [f"pct_{c.lower()}" for c in cat_pct.columns]
        return out.join(cat_pct)

    print("\n" + "=" * 80)
    print("MARGINAL WIN RATES — by MODEL (pooled across configs)")
    print("=" * 80)
    by_model = _faction_marginal("model").reindex(model_order)
    display(
        by_model.style
        .format({c: "{:.2%}" for c in by_model.columns if c != "n_games"})
        .format({"n_games": "{:d}"})
        .set_caption("Per-model win rates and category mix")
    )

    print("\n" + "=" * 80)
    print("MARGINAL WIN RATES — by CONFIG (pooled across models)")
    print("=" * 80)
    by_cfg = _faction_marginal("config").reindex(config_order)
    display(
        by_cfg.style
        .format({c: "{:.2%}" for c in by_cfg.columns if c != "n_games"})
        .format({"n_games": "{:d}"})
        .set_caption("Per-configuration win rates and category mix")
    )

    # ── Persist to disk so downstream artifacts are reproducible ──
    # Tag artifacts with the date of the underlying experiment logs
    # (2026-04-17 sweep — yesterday's runs in expt-logs/2026-04-17_exp_*).
    _date = "2026-04-17"
    out_long = os.path.join(RESULTS_PATH, f"{_date}_game_outcomes_long.csv")
    out_faction = os.path.join(RESULTS_PATH,
                               f"{_date}_win_rates_model_x_config.csv")
    out_categ = os.path.join(RESULTS_PATH,
                             f"{_date}_win_categories_model_x_config.csv")
    df_wins.to_csv(out_long, index=False)
    faction_tbl.to_csv(out_faction, index=False)
    cat_share.reset_index().to_csv(out_categ, index=False)
    print("\nSaved:")
    print(f"  {out_long}")
    print(f"  {out_faction}")
    print(f"  {out_categ}")

---
### 6e. Theory-of-Mind Radar — 6-axis profile per role × model

Replaces the older 2-axis Pareto scatter with a **hexagonal radar** so a single
shape captures each model's strengths and weaknesses across the dimensions
that matter for the role.

* **Crewmate axes:** \(1 - \text{detection\_MSE}\), `objective_viability`,
  `alibi_grounding`, `alibi_corroboration`, `faction_consensus`,
  `social_influence`. (Detection is inverted so outward = better-calibrated.)
* **Impostor axes:** `deceptive_efficacy`, `objective_viability`,
  \(1 - \text{alibi\_grounding}\) (alibi opacity — higher = better lying),
  \(1 - \text{belief\_volatility}\) (stability under pressure),
  `faction_consensus`, `social_influence`.

Each axis is **min-max normalized within the set of models in this run**, so
the polygons are commensurable but the scale is *relative*, not absolute.
Models that are non-dominated across all 6 normalized axes are drawn with a
bold stroke and marked **\*** in the legend (the 6-D Pareto frontier).

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 6-axis radar (hexagon) profile per role × model.
#
# Replaces the older 2-axis Pareto scatter with a full 6-metric
# radar: a single shape captures each model's strengths and
# weaknesses across the dimensions that matter for that role.
#
# Per-axis min-max normalization (within each role's set of axes,
# across the *models present in this run*) makes the polygons
# commensurable. Each axis spans [0, 1], so axis lengths are
# *relative* — comparing models on the same plot, NOT against an
# absolute scale.
#
# Axes are oriented so OUTWARD = BETTER for that role:
#   Crewmate:  ↑ (1 − detection_MSE) / objective_viability /
#              alibi_grounding / alibi_corroboration /
#              faction_consensus / social_influence
#   Impostor:  ↑ deceptive_efficacy / objective_viability /
#              (1 − alibi_grounding)  ← high = good lying
#              (1 − belief_volatility) ← high = stable under pressure
#              faction_consensus / social_influence
#
# Models that are non-dominated across all 6 normalized axes get a
# bold stroke + asterisk in the legend (the 6-D Pareto frontier).
# ══════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# (column_name, display_label, mode)
#   mode = "raw"    → use mean as-is
#   mode = "invert" → use 1 − mean (so outward = better)
CREWMATE_RADAR_AXES = [
    ("detection_accuracy",  "Detection\n(1−MSE)",         "invert"),
    ("objective_viability", "Objective\nviability",       "raw"),
    ("alibi_grounding",     "Alibi\ngrounding",           "raw"),
    ("alibi_corroboration", "Alibi\ncorroboration",       "raw"),
    ("faction_consensus",   "Faction\nconsensus",         "raw"),
    ("social_influence",    "Social\ninfluence",          "raw"),
]

IMPOSTOR_RADAR_AXES = [
    ("deceptive_efficacy",  "Deceptive\nefficacy",        "raw"),
    ("objective_viability", "Objective\nviability",       "raw"),
    ("alibi_grounding",     "Alibi\nopacity\n(1−grounding)", "invert"),
    ("belief_volatility",   "Stability\n(1−volatility)",  "invert"),
    ("faction_consensus",   "Faction\nconsensus",         "raw"),
    ("social_influence",    "Social\ninfluence",          "raw"),
]


def _aggregate_for_radar(
    df: pd.DataFrame, axes_spec: list, role_id: int
) -> pd.DataFrame:
    """Return a (model × axis) matrix of mean values, one row per model.

    Applies the ``invert`` transform (``1 - x``) to axes flagged that way
    so that *outward = better* on every axis. Columns come back in
    ``axes_spec`` order, with two metadata columns prepended.
    """
    sub = df[df["identity"] == role_id]
    if sub.empty:
        return pd.DataFrame()
    rows = []
    for m, g in sub.groupby("model"):
        row = {"model": m, "n": len(g)}
        for col, _label, mode in axes_spec:
            if col not in g.columns:
                row[col] = np.nan
                continue
            v = g[col].mean()
            if mode == "invert":
                v = 1.0 - v
            row[col] = v
        rows.append(row)
    out = pd.DataFrame(rows)
    # Preserve MODELS_TO_RUN order so the legend is deterministic across
    # re-runs (rather than alphabetic, which would mix nano with mini).
    if "model" in out.columns:
        order = [m for m in MODELS_TO_RUN if m in out["model"].tolist()]
        out = pd.concat(
            [out[out["model"] == m] for m in order], ignore_index=True
        )
    return out


def _normalize_columns(mat: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Per-axis min-max normalize to [0, 1] across rows (= models)."""
    out = mat.copy()
    for c in cols:
        x = mat[c].astype(float)
        if x.notna().sum() == 0:
            out[c] = 0.0
            continue
        lo, hi = float(x.min()), float(x.max())
        if hi - lo < 1e-9:
            out[c] = 0.5  # all-equal axis: neutral midline
        else:
            out[c] = (x - lo) / (hi - lo)
    return out


def _pareto_set_max(M: np.ndarray) -> list[int]:
    """Indices nondominated when MAXIMIZING every column. n×d, d≥1."""
    n = M.shape[0]
    out: list[int] = []
    for i in range(n):
        dominated = False
        for j in range(n):
            if i == j:
                continue
            if np.all(M[j] >= M[i]) and np.any(M[j] > M[i]):
                dominated = True
                break
        if not dominated:
            out.append(i)
    return out


def _plot_radar(
    ax, mat_norm: pd.DataFrame, axes_spec: list,
    title: str, pareto_idx: set,
) -> None:
    """Render one radar panel onto a pre-created polar Axes."""
    cols = [c for c, _, _ in axes_spec]
    labels = [lab for _, lab, _ in axes_spec]
    n_axes = len(cols)
    angles = np.linspace(0, 2 * np.pi, n_axes, endpoint=False).tolist()
    angles += angles[:1]  # close the loop

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["", "0.5", "", "1.0"], fontsize=7, color="0.45")
    ax.set_ylim(0, 1.05)
    ax.set_rlabel_position(0)
    ax.grid(color="0.85", lw=0.6)

    cmap = plt.get_cmap("tab10")
    for i, (_, row) in enumerate(mat_norm.iterrows()):
        vals = [float(row[c]) if pd.notna(row[c]) else 0.0 for c in cols]
        vals += vals[:1]  # close the loop
        color = cmap(i % 10)
        is_pareto = i in pareto_idx
        lw = 2.2 if is_pareto else 1.2
        alpha_fill = 0.18 if is_pareto else 0.06
        label = str(row["model"]) + (" *" if is_pareto else "")
        ax.plot(angles, vals, color=color, lw=lw, label=label)
        ax.fill(angles, vals, color=color, alpha=alpha_fill)

    ax.set_title(title, fontsize=11, pad=18)
    ax.legend(
        loc="upper right", bbox_to_anchor=(1.45, 1.10),
        fontsize=8, frameon=False,
    )


if df_metrics.empty or "model" not in df_metrics.columns:
    print(
        "df_metrics is empty or missing \"model\" column — run the metrics "
        "sweep cell first."
    )
else:
    crew_raw = _aggregate_for_radar(df_metrics, CREWMATE_RADAR_AXES, role_id=0)
    imp_raw  = _aggregate_for_radar(df_metrics, IMPOSTOR_RADAR_AXES,  role_id=1)

    crew_cols = [c for c, _, _ in CREWMATE_RADAR_AXES]
    imp_cols  = [c for c, _, _ in IMPOSTOR_RADAR_AXES]

    crew_norm = (
        _normalize_columns(crew_raw, crew_cols) if not crew_raw.empty else crew_raw
    )
    imp_norm = (
        _normalize_columns(imp_raw, imp_cols) if not imp_raw.empty else imp_raw
    )

    crew_pareto = (
        set(_pareto_set_max(crew_norm[crew_cols].to_numpy(dtype=float)))
        if not crew_norm.empty else set()
    )
    imp_pareto = (
        set(_pareto_set_max(imp_norm[imp_cols].to_numpy(dtype=float)))
        if not imp_norm.empty else set()
    )

    fig = plt.figure(figsize=(13.5, 5.6))
    ax_crew = fig.add_subplot(1, 2, 1, projection="polar")
    ax_imp  = fig.add_subplot(1, 2, 2, projection="polar")

    if crew_norm.empty:
        ax_crew.text(0, 0, "No crewmate data", ha="center")
    else:
        _plot_radar(
            ax_crew, crew_norm, CREWMATE_RADAR_AXES,
            "Crewmate — outward = better\n(per-axis min-max across models; * = 6-D Pareto)",
            crew_pareto,
        )

    if imp_norm.empty:
        ax_imp.text(0, 0, "No impostor data", ha="center")
    else:
        _plot_radar(
            ax_imp, imp_norm, IMPOSTOR_RADAR_AXES,
            "Impostor — outward = better\n(per-axis min-max across models; * = 6-D Pareto)",
            imp_pareto,
        )

    fig.suptitle(
        "Theory-of-Mind Radar — 6-axis profile per role × model",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    plt.show()

    print("\nRaw axis values (Crewmate; detection inverted as 1−MSE):")
    if not crew_raw.empty:
        cols_show = ["model", "n"] + crew_cols
        print(
            crew_raw[cols_show]
            .to_string(index=False, float_format="{:.4f}".format)
        )
    print(
        "\nRaw axis values (Impostor; alibi_grounding & belief_volatility "
        "shown as 1−x):"
    )
    if not imp_raw.empty:
        cols_show = ["model", "n"] + imp_cols
        print(
            imp_raw[cols_show]
            .to_string(index=False, float_format="{:.4f}".format)
        )


---
## 7. Cleanup

In [ ]:
# Optionally kill every vLLM server launched by this notebook.
# (Servers started outside the notebook — e.g. left over from a
# previous session that hit the port-in-use branch in cell 7 — are
# not in `vllm_processes` and won't be touched.)
_running = {
    k: p for k, p in vllm_processes.items()
    if p is not None and p.poll() is None
}
if _running:
    for _key, _proc in _running.items():
        print(f"Shutting down vLLM server [{_key}] (PID: {_proc.pid})...")
        try:
            os.killpg(os.getpgid(_proc.pid), signal.SIGTERM)
            _proc.wait(timeout=10)
            print(f"  [{_key}] stopped.")
        except Exception as e:
            print(f"  [{_key}] failed to stop cleanly: {e}")
else:
    print("No vLLM servers were started by this notebook (or all already stopped).")

---
## 13. Belief Calibration — Verbalized vs. Token-Logprob Distributions

Compares the model's **verbalized** belief probabilities (numbers it wrote into the
`belief_distribution` JSON) against the **logprob-derived** distribution (built from
the next-token Yes/No posterior in `belief_distribution_logprobs`) — both scored
against the ground-truth Impostor labels from each game's `summary.json`.

A well-calibrated forecaster's reliability curve hugs the diagonal. We expect
the verbalized line to cluster around round values (0.1 / 0.5 / 0.9), while the
logprob line should sit much closer to the diagonal. Reported metrics:

- **ECE** (Expected Calibration Error) — weighted mean gap between predicted
  probability and observed frequency across bins. Lower is better.
- **Brier score** — mean squared error between probability and outcome (0/1).
  Lower is better.


In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np

from evaluations.metrics_calculator import _load_all_summaries

# ── 1. Discover all experiments with epistemic logs ─────────────────────────
epi_files = sorted(glob.glob(os.path.join(LOGS_PATH, "*", "epistemic-states.jsonl")))
print(f"Found {len(epi_files)} experiments with epistemic logs.")

# ── 2. Aggregate (predicted_prob, ground_truth) pairs across all games ──────
# We store both a pooled (cross-config) sample and per-config samples so
# that the per-config reliability plot below isn't confounded by the
# different impostor priors of each config:
#   4C_1I  → prior P(opp is Impostor) = 1/4 = 0.25
#   4C_2I  → 2/4 = 0.50
#   5C_1I  → 1/5 = 0.20
#   5C_2I  → 2/5 = 0.40
def _config_name(num_players: int, num_impostors: int) -> str:
    """Recover the canonical config tag (matches GAME_CONFIGS keys)."""
    n_crew = num_players - num_impostors
    return f"{n_crew}C_{num_impostors}I"

verbalized_p, verbalized_y = [], []
logprob_p,    logprob_y    = [], []
# Per-config samples (paired): {cfg_name: {"verb_p":[], "lp_p":[], "y":[]}}
per_cfg: "dict[str, dict[str, list]]" = {}
n_with_lp = 0
n_total_snaps = 0

for epi_path in epi_files:
    expt_dir = os.path.dirname(epi_path)
    summary_path = os.path.join(expt_dir, "summary.json")
    # ``summary.json`` is JSONL (one game record per line) — parallel
    # games each append independently, so the file holds N concatenated
    # objects, not a single one. Use the helper that handles that.
    summaries = _load_all_summaries(summary_path)
    if not summaries:
        continue

    impostors_by_game: dict[str, set[str]] = {}
    cfg_by_game: dict[str, str] = {}
    for summary in summaries:
        game_key = next(
            (k for k in summary if isinstance(k, str) and k.startswith("Game")),
            None,
        )
        if not game_key:
            continue
        gt = _extract_ground_truth_from_summary(summary, game_key)
        if gt is None:
            continue
        impostors_by_game[game_key] = gt.impostor_names
        cfg = summary[game_key].get("config", {}) or {}
        np_, ni_ = cfg.get("num_players"), cfg.get("num_impostors")
        if np_ and ni_:
            cfg_by_game[game_key] = _config_name(int(np_), int(ni_))

    if not impostors_by_game:
        continue

    parser = EpistemicLogParser()
    snapshots = parser.parse_file(epi_path)
    n_total_snaps += len(snapshots)

    # If snapshots have game_index, use it; otherwise fall back to the union
    # of impostor names (good enough — color suffixes are unique per game)
    union_impostors: set[str] = set().union(*impostors_by_game.values())
    # When game_index is missing we don't know the per-game config, so we
    # only attribute snapshots whose config is unambiguous (single-config
    # experiment dirs — true for our sweep, since each dir is one config).
    unique_cfgs = set(cfg_by_game.values())
    fallback_cfg = next(iter(unique_cfgs)) if len(unique_cfgs) == 1 else None

    # ── PAIRED comparison ──────────────────────────────────────────────
    # We only count predictions for which BOTH a verbalized probability
    # AND a logprob-derived probability exist for the same (snapshot,
    # opponent) pair, so the two reliability curves are computed over
    # an identical sample. Otherwise the verbalized curve aggregates
    # ~5× more predictions than the logprob curve and any difference in
    # ECE would conflate calibration with sample mix.
    for s in snapshots:
        if not s.belief_distribution_logprobs:
            continue  # snapshot lacks the logprob signal — skip entirely

        gi = s.game_index
        impostor_set = (
            impostors_by_game.get(gi)
            if gi and gi in impostors_by_game
            else union_impostors
        )
        cfg_name = cfg_by_game.get(gi, fallback_cfg)

        common_keys = (
            set(s.belief_distribution.keys())
            & set(s.belief_distribution_logprobs.keys())
        )
        if not common_keys:
            continue

        n_with_lp += 1
        cfg_bucket = per_cfg.setdefault(
            cfg_name or "unknown",
            {"verb_p": [], "lp_p": [], "y": []},
        )
        for p_name in common_keys:
            y = 1.0 if p_name in impostor_set else 0.0
            vp = float(s.belief_distribution[p_name])
            lpp = float(s.belief_distribution_logprobs[p_name])
            verbalized_p.append(vp);   verbalized_y.append(y)
            logprob_p.append(lpp);     logprob_y.append(y)
            cfg_bucket["verb_p"].append(vp)
            cfg_bucket["lp_p"].append(lpp)
            cfg_bucket["y"].append(y)

verbalized_p = np.asarray(verbalized_p)
verbalized_y = np.asarray(verbalized_y)
logprob_p    = np.asarray(logprob_p)
logprob_y    = np.asarray(logprob_y)
for cfg_bucket in per_cfg.values():
    for k in cfg_bucket:
        cfg_bucket[k] = np.asarray(cfg_bucket[k], dtype=float)

# Sanity: by construction the two arrays must align row-for-row.
assert verbalized_p.shape == logprob_p.shape == verbalized_y.shape == logprob_y.shape
assert np.array_equal(verbalized_y, logprob_y), \
    "paired-sample invariant broken — y vectors differ between curves"

print(f"Total snapshots parsed                  : {n_total_snaps}")
print(f"Snapshots with BOTH signals (paired)    : {n_with_lp}")
print(f"Paired (snapshot, opponent) predictions : {len(verbalized_p)}")
print(f"  base rate (P[opponent is Impostor])   : {verbalized_y.mean():.3f}")

# ── 3. Calibration metrics ──────────────────────────────────────────────────
def _reliability(p: np.ndarray, y: np.ndarray, n_bins: int = 10):
    """Bin probabilities and return (mean_p, mean_y, weight) per non-empty bin."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(p, bins, right=False) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = idx == b
        n = mask.sum()
        if n == 0:
            continue
        rows.append((p[mask].mean(), y[mask].mean(), n / len(p)))
    return rows

def _ece(p: np.ndarray, y: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error (weighted mean |mean_p - mean_y| over bins)."""
    if len(p) == 0:
        return float("nan")
    rows = _reliability(p, y, n_bins=n_bins)
    return float(sum(abs(mp - my) * w for mp, my, w in rows))

def _brier(p: np.ndarray, y: np.ndarray) -> float:
    return float(np.mean((p - y) ** 2)) if len(p) else float("nan")

stats = {
    "Verbalized": {
        "n":     len(verbalized_p),
        "ECE":   _ece(verbalized_p, verbalized_y),
        "Brier": _brier(verbalized_p, verbalized_y),
        "Base rate (y=1)": float(verbalized_y.mean()) if len(verbalized_y) else float("nan"),
    },
    "Logprob-derived": {
        "n":     len(logprob_p),
        "ECE":   _ece(logprob_p, logprob_y),
        "Brier": _brier(logprob_p, logprob_y),
        "Base rate (y=1)": float(logprob_y.mean()) if len(logprob_y) else float("nan"),
    },
}

print("\nCalibration summary")
print("-" * 60)
for source, m in stats.items():
    print(f"{source:18s}  n={m['n']:>5d}  "
          f"ECE={m['ECE']:.4f}  Brier={m['Brier']:.4f}  "
          f"base_rate={m['Base rate (y=1)']:.3f}")

# ── 4. Reliability diagram + probability histograms ─────────────────────────
if len(verbalized_p) == 0 and len(logprob_p) == 0:
    print("\nNo belief data to plot.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

    # Left: reliability diagram
    ax = axes[0]
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    for label, p, y, color in [
        ("Verbalized",      verbalized_p, verbalized_y, "tab:orange"),
        ("Logprob-derived", logprob_p,    logprob_y,    "tab:blue"),
    ]:
        if len(p) == 0:
            continue
        rows = _reliability(p, y, n_bins=10)
        if not rows:
            continue
        xs = [r[0] for r in rows]
        ys = [r[1] for r in rows]
        ws = [r[2] for r in rows]
        ax.plot(xs, ys, "o-", color=color, label=f"{label}  (ECE={_ece(p, y):.3f})")
        ax.scatter(xs, ys, s=[max(20, 600 * w) for w in ws],
                   color=color, alpha=0.25, zorder=1)
    ax.set_xlabel("Predicted P(Impostor)")
    ax.set_ylabel("Observed frequency of Impostor")
    ax.set_title("Belief calibration (reliability diagram)")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left")

    # Right: probability histograms
    ax = axes[1]
    bins = np.linspace(0, 1, 21)
    if len(verbalized_p):
        ax.hist(verbalized_p, bins=bins, alpha=0.5, color="tab:orange",
                label=f"Verbalized (n={len(verbalized_p)})", density=True)
    if len(logprob_p):
        ax.hist(logprob_p, bins=bins, alpha=0.5, color="tab:blue",
                label=f"Logprob-derived (n={len(logprob_p)})", density=True)
    ax.set_xlabel("Predicted P(Impostor)")
    ax.set_ylabel("Density")
    ax.set_title("Distribution of predicted probabilities")
    ax.legend(loc="upper center")
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out_png = os.path.join(RESULTS_PATH, "belief_calibration.png")
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nSaved figure to: {out_png}")


# ── 5. Per-config calibration ──────────────────────────────────────────────
# The pooled diagram averages across configs with very different impostor
# priors (0.20 / 0.25 / 0.40 / 0.50). A non-informative agent that just
# emits the prior would land at a *different* point on each config's
# diagonal, so pooling hides whether the agent is genuinely uplift-ing
# the prior. We replot one panel per config and overlay each config's
# prior as a red cross-hair so the reader can see at a glance whether the
# curve actually rises *above* the prior in the high-confidence bins.
CFG_ORDER = ["4C_1I", "4C_2I", "5C_1I", "5C_2I"]
present = [c for c in CFG_ORDER if c in per_cfg and len(per_cfg[c]["y"]) > 0]
extras  = [c for c in per_cfg if c not in CFG_ORDER and len(per_cfg[c]["y"]) > 0]
present.extend(extras)

if present:
    n = len(present)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 4.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, cfg in enumerate(present):
        ax = axes[i]
        d = per_cfg[cfg]
        prior = float(d["y"].mean())  # empirical = exact for fixed config
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
        ax.axhline(prior, color="tab:red", ls=":", alpha=0.6,
                   label=f"Prior = {prior:.2f}")
        ax.axvline(prior, color="tab:red", ls=":", alpha=0.6)
        for label, p, color in [
            ("Verbalized",      d["verb_p"], "tab:orange"),
            ("Logprob-derived", d["lp_p"],   "tab:blue"),
        ]:
            if len(p) == 0:
                continue
            rows = _reliability(p, d["y"], n_bins=10)
            if not rows:
                continue
            xs = [r[0] for r in rows]
            ys = [r[1] for r in rows]
            ws = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", color=color,
                    label=f"{label}  (ECE={_ece(p, d['y']):.3f})")
            ax.scatter(xs, ys,
                       s=[max(20, 600 * w) for w in ws],
                       color=color, alpha=0.25, zorder=1)
        ax.set_title(f"{cfg}    n={len(d['y']):,}    prior={prior:.2f}")
        ax.set_xlabel("Predicted P(Impostor)")
        ax.set_ylabel("Observed frequency of Impostor")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper left", fontsize=8)

    for j in range(len(present), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    out_png = os.path.join(RESULTS_PATH, "belief_calibration_by_config.png")
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to: {out_png}")


# ── 6. Lift-over-prior helper (used by the unpaired table in section 9) ────
# For low-base-rate problems ECE alone can be misleading: a constant
# predictor that always emits the prior achieves ECE ≈ 0 but is useless
# (no discrimination). LIFT measures discrimination directly:
#
#     lift(p)  =  P(impostor | prediction in bin)  /  P(impostor)
#
# - lift = 1  → bin gives no information beyond the base rate
# - lift > 1  → opponents in this confidence bin are MORE likely than
#               baseline to actually be impostors (good — the agent's
#               confident calls really do select impostors)
# - lift < 1  → less likely than baseline (good for low-confidence bins;
#               means the agent successfully clears innocents there)
#
# We focus on the **top decile** (`p ≥ 0.9`) — the "I'm sure" bin — because
# that's the one that drives ejection votes. The actual table is built
# in section 9 from the unpaired per-signal aggregations so that each
# entry uses the maximum data available for that signal.
def _top_decile_lift(p: np.ndarray, y: np.ndarray, threshold: float = 0.9):
    if len(p) == 0:
        return float("nan"), 0, float("nan")
    mask = p >= threshold
    n_top = int(mask.sum())
    if n_top == 0:
        return float("nan"), 0, float("nan")
    base = float(y.mean())
    precision = float(y[mask].mean())  # = P(impostor | p >= threshold)
    lift = precision / base if base > 0 else float("nan")
    return lift, n_top, precision

import pandas as pd


# ── 7. Unpaired per-signal calibration ─────────────────────────────────────
# The paired analysis above ensures a fair head-to-head ECE comparison
# but discards ~80% of verbalized predictions (snapshots from API models
# where we don't yet collect logprobs). For describing each signal in
# isolation we want its FULL sample. Below we re-aggregate without the
# pairing constraint and produce one figure per signal.
all_verb_per_cfg: "dict[str, dict[str, list]]" = {}
all_logp_per_cfg: "dict[str, dict[str, list]]" = {}

for epi_path in epi_files:
    expt_dir = os.path.dirname(epi_path)
    summary_path = os.path.join(expt_dir, "summary.json")
    summaries = _load_all_summaries(summary_path)
    if not summaries:
        continue

    impostors_by_game: dict[str, set[str]] = {}
    cfg_by_game: dict[str, str] = {}
    for summary in summaries:
        game_key = next(
            (k for k in summary if isinstance(k, str) and k.startswith("Game")),
            None,
        )
        if not game_key:
            continue
        gt = _extract_ground_truth_from_summary(summary, game_key)
        if gt is None:
            continue
        impostors_by_game[game_key] = gt.impostor_names
        cfg = summary[game_key].get("config", {}) or {}
        np_, ni_ = cfg.get("num_players"), cfg.get("num_impostors")
        if np_ and ni_:
            cfg_by_game[game_key] = _config_name(int(np_), int(ni_))

    if not impostors_by_game:
        continue

    union_impostors: set[str] = set().union(*impostors_by_game.values())
    unique_cfgs = set(cfg_by_game.values())
    fallback_cfg = next(iter(unique_cfgs)) if len(unique_cfgs) == 1 else None

    parser = EpistemicLogParser()
    snapshots = parser.parse_file(epi_path)

    for s in snapshots:
        gi = s.game_index
        impostor_set = (
            impostors_by_game.get(gi)
            if gi and gi in impostors_by_game
            else union_impostors
        )
        cfg_name_ = cfg_by_game.get(gi, fallback_cfg) or "unknown"

        if s.belief_distribution:
            bucket = all_verb_per_cfg.setdefault(
                cfg_name_, {"p": [], "y": []}
            )
            for p_name, prob in s.belief_distribution.items():
                bucket["p"].append(float(prob))
                bucket["y"].append(1.0 if p_name in impostor_set else 0.0)

        if s.belief_distribution_logprobs:
            bucket = all_logp_per_cfg.setdefault(
                cfg_name_, {"p": [], "y": []}
            )
            for p_name, prob in s.belief_distribution_logprobs.items():
                bucket["p"].append(float(prob))
                bucket["y"].append(1.0 if p_name in impostor_set else 0.0)

for store in (all_verb_per_cfg, all_logp_per_cfg):
    for cfg_name_ in store:
        store[cfg_name_] = {
            k: np.asarray(v, dtype=float) for k, v in store[cfg_name_].items()
        }


def _plot_per_config_reliability(
    per_cfg_dict: "dict[str, dict[str, np.ndarray]]",
    title: str,
    color: str,
    fname: str,
) -> None:
    """Render a 2×2 grid of reliability diagrams (one panel per config)."""
    cfgs = [c for c in CFG_ORDER if c in per_cfg_dict and len(per_cfg_dict[c]["y"]) > 0]
    cfgs.extend(
        c for c in per_cfg_dict
        if c not in CFG_ORDER and len(per_cfg_dict[c]["y"]) > 0
    )
    if not cfgs:
        print(f"[{title}] no data — skipping.")
        return

    n = len(cfgs)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(11, 4.6 * nrows))
    axes = np.atleast_1d(axes).ravel()

    for i, cfg in enumerate(cfgs):
        ax = axes[i]
        d = per_cfg_dict[cfg]
        prior = float(d["y"].mean())
        ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
        ax.axhline(prior, color="tab:red", ls=":", alpha=0.6,
                   label=f"Prior = {prior:.2f}")
        ax.axvline(prior, color="tab:red", ls=":", alpha=0.6)

        rows = _reliability(d["p"], d["y"], n_bins=10)
        if rows:
            xs = [r[0] for r in rows]
            ys = [r[1] for r in rows]
            ws = [r[2] for r in rows]
            ax.plot(xs, ys, "o-", color=color,
                    label=f"ECE={_ece(d['p'], d['y']):.3f}")
            ax.scatter(xs, ys,
                       s=[max(20, 600 * w) for w in ws],
                       color=color, alpha=0.25, zorder=1)
        ax.set_title(f"{cfg}    n={len(d['y']):,}    prior={prior:.2f}")
        ax.set_xlabel("Predicted P(Impostor)")
        ax.set_ylabel("Observed frequency of Impostor")
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.legend(loc="upper left", fontsize=8)

    for j in range(len(cfgs), len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=13, y=1.005)
    plt.tight_layout()
    out_png = os.path.join(RESULTS_PATH, fname)
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to: {out_png}")


_plot_per_config_reliability(
    all_verb_per_cfg,
    "Verbalized belief calibration — all predictions",
    "tab:orange",
    "belief_calibration_verbalized_all.png",
)
_plot_per_config_reliability(
    all_logp_per_cfg,
    "Logprob-derived belief calibration — all predictions",
    "tab:blue",
    "belief_calibration_logprob_all.png",
)


# ── 8. Combined overlay (all configs on one axis) ─────────────────────────
# A single-panel reliability diagram per signal, with one curve per config.
# Useful for spotting how calibration drifts as the impostor prior varies
# (low-prior configs are above the diagonal at low p, high-prior configs
# tend to be below it). Each config's prior is rendered as a small "x" on
# the diagonal so you can see at a glance where its neutral point sits.
_CFG_COLORS = {
    "4C_1I": "tab:blue",
    "4C_2I": "tab:orange",
    "5C_1I": "tab:green",
    "5C_2I": "tab:red",
}


def _plot_combined_reliability(
    per_cfg_dict: "dict[str, dict[str, np.ndarray]]",
    title: str,
    fname: str,
) -> None:
    """One axis, one curve per config, with priors marked on the diagonal."""
    cfgs = [c for c in CFG_ORDER if c in per_cfg_dict and len(per_cfg_dict[c]["y"]) > 0]
    cfgs.extend(
        c for c in per_cfg_dict
        if c not in CFG_ORDER and len(per_cfg_dict[c]["y"]) > 0
    )
    if not cfgs:
        print(f"[{title}] no data — skipping.")
        return

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")

    for cfg in cfgs:
        d = per_cfg_dict[cfg]
        color = _CFG_COLORS.get(cfg, None)
        prior = float(d["y"].mean())
        rows = _reliability(d["p"], d["y"], n_bins=10)
        if not rows:
            continue
        xs = [r[0] for r in rows]
        ys = [r[1] for r in rows]
        ws = [r[2] for r in rows]
        ax.plot(
            xs, ys, "o-", color=color,
            label=(
                f"{cfg}  (n={len(d['y']):,}, prior={prior:.2f}, "
                f"ECE={_ece(d['p'], d['y']):.3f})"
            ),
        )
        ax.scatter(xs, ys,
                   s=[max(20, 600 * w) for w in ws],
                   color=color, alpha=0.20, zorder=1)
        # Mark this config's prior on the diagonal — its "non-informative" point.
        ax.plot(prior, prior, marker="x", color=color, markersize=11,
                markeredgewidth=2.2, zorder=3)

    ax.set_xlabel("Predicted P(Impostor)")
    ax.set_ylabel("Observed frequency of Impostor")
    ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    out_png = os.path.join(RESULTS_PATH, fname)
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to: {out_png}")


_plot_combined_reliability(
    all_verb_per_cfg,
    "Verbalized belief calibration — all configs",
    "belief_calibration_verbalized_combined.png",
)
_plot_combined_reliability(
    all_logp_per_cfg,
    "Logprob-derived belief calibration — all configs",
    "belief_calibration_logprob_combined.png",
)


# ── 8b. Pooled single-line reliability per signal ────────────────────────
# Concatenate every config's predictions into one big sample and draw a
# single reliability curve. This is the "headline" calibration figure for
# each signal — one number (ECE), one curve, no per-config breakdown.
# Useful for the report's executive summary table / one-line claim.
def _pooled_arrays(per_cfg_dict):
    p_all, y_all = [], []
    for d in per_cfg_dict.values():
        if len(d["y"]) == 0:
            continue
        p_all.append(d["p"])
        y_all.append(d["y"])
    if not p_all:
        return np.array([]), np.array([])
    return np.concatenate(p_all), np.concatenate(y_all)


def _plot_pooled_single_line(
    per_cfg_dict: "dict[str, dict[str, np.ndarray]]",
    title: str,
    color: str,
    fname: str,
) -> None:
    p, y = _pooled_arrays(per_cfg_dict)
    if len(p) == 0:
        print(f"[{title}] no data — skipping.")
        return
    prior = float(y.mean())
    ece_val = _ece(p, y)
    rows = _reliability(p, y, n_bins=10)

    fig, ax = plt.subplots(figsize=(7.0, 6.0))
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect calibration")
    if rows:
        xs = [r[0] for r in rows]
        ys = [r[1] for r in rows]
        ws = [r[2] for r in rows]
        ax.plot(xs, ys, "o-", color=color,
                label=f"Pooled  (n={len(p):,}, ECE={ece_val:.3f})")
        ax.scatter(xs, ys,
                   s=[max(20, 600 * w) for w in ws],
                   color=color, alpha=0.25, zorder=1)
    # Pooled prior cross-hair (the average of the four configs' priors,
    # weighted by their unpaired sample sizes).
    ax.axhline(prior, color="tab:red", ls=":", alpha=0.6,
               label=f"Pooled prior = {prior:.2f}")
    ax.axvline(prior, color="tab:red", ls=":", alpha=0.6)

    ax.set_xlabel("Predicted P(Impostor)")
    ax.set_ylabel("Observed frequency of Impostor")
    ax.set_title(title)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=10)
    plt.tight_layout()
    out_png = os.path.join(RESULTS_PATH, fname)
    plt.savefig(out_png, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Saved figure to: {out_png}")


_plot_pooled_single_line(
    all_verb_per_cfg,
    "Verbalized belief calibration — pooled across all configs",
    "tab:orange",
    "belief_calibration_verbalized_pooled.png",
)
_plot_pooled_single_line(
    all_logp_per_cfg,
    "Logprob-derived belief calibration — pooled across all configs",
    "tab:blue",
    "belief_calibration_logprob_pooled.png",
)


# ── 9. Unpaired lift table (uses all available data per signal) ───────────
# Same lift-over-prior analysis as section 6, but built from the FULL
# unpaired samples (~5,370 verbalized, ~1,075 logprob) so each cell of
# the table is computed with the maximum data available for that signal.
unpaired_rows = []
for cfg in CFG_ORDER + [c for c in set(all_verb_per_cfg) | set(all_logp_per_cfg)
                       if c not in CFG_ORDER]:
    if cfg not in all_verb_per_cfg and cfg not in all_logp_per_cfg:
        continue
    v = all_verb_per_cfg.get(cfg)
    l = all_logp_per_cfg.get(cfg)

    # Each signal carries its own y vector (the paired-sample assert
    # doesn't apply here), and its own base rate.
    if v is not None and len(v["y"]):
        v_base = float(v["y"].mean())
        v_ece = _ece(v["p"], v["y"])
        v_lift, v_n_top, v_prec = _top_decile_lift(v["p"], v["y"])
        v_n = len(v["y"])
    else:
        v_base = v_ece = v_lift = v_prec = float("nan")
        v_n = v_n_top = 0

    if l is not None and len(l["y"]):
        l_base = float(l["y"].mean())
        l_ece = _ece(l["p"], l["y"])
        l_lift, l_n_top, l_prec = _top_decile_lift(l["p"], l["y"])
        l_n = len(l["y"])
    else:
        l_base = l_ece = l_lift = l_prec = float("nan")
        l_n = l_n_top = 0

    unpaired_rows.append({
        "config":              cfg,
        "verb_n":              v_n,
        "verb_prior":          v_base,
        "verb_ECE":            v_ece,
        "verb_precision@p≥0.9": v_prec,
        "verb_lift@p≥0.9":     v_lift,
        "verb_n_top":          v_n_top,
        "logp_n":              l_n,
        "logp_prior":          l_base,
        "logp_ECE":            l_ece,
        "logp_precision@p≥0.9": l_prec,
        "logp_lift@p≥0.9":     l_lift,
        "logp_n_top":          l_n_top,
    })

if unpaired_rows:
    unpaired_lift_df = pd.DataFrame(unpaired_rows).set_index("config")
    fmt = {
        "verb_prior":            "{:.2f}",
        "verb_ECE":              "{:.3f}",
        "verb_precision@p≥0.9":  "{:.2f}",
        "verb_lift@p≥0.9":       "{:.2f}",
        "logp_prior":            "{:.2f}",
        "logp_ECE":              "{:.3f}",
        "logp_precision@p≥0.9":  "{:.2f}",
        "logp_lift@p≥0.9":       "{:.2f}",
    }
    print("\nPer-config calibration & lift over prior  (all available "
          "data per signal — UNPAIRED)")
    print("-" * 60)
    try:
        from IPython.display import display
        display(unpaired_lift_df.style.format(fmt).set_caption(
            "prior = P(opponent is Impostor) under random guessing for this "
            "config (= n_impostors / n_living_opponents). "
            "ECE = expected calibration error: sample-weighted mean gap "
            "between predicted probability and observed frequency across 10 "
            "bins; 0 = perfectly calibrated. "
            "precision@p\u22650.9 = fraction of opponents the agent flagged "
            "with probability \u2265 0.9 who really were Impostors (i.e. how "
            "often a confident accusation is correct). "
            "lift@p\u22650.9 = precision@p\u22650.9 / prior; >1 means "
            "confident calls select Impostors above the random-chance rate "
            "for this config (1.0 = no information; 4.0 = 4\u00d7 better than "
            "random)."
        ))
    except Exception:
        print(unpaired_lift_df.to_string(
            formatters={k: v.format for k, v in fmt.items()}
        ))
    out_csv = os.path.join(
        RESULTS_PATH, "belief_calibration_by_config_unpaired.csv"
    )
    unpaired_lift_df.to_csv(out_csv)
    print(f"Saved table to:  {out_csv}")


---
## 14. Belief Calibration — Per-Model (across all configs combined)

For each model in the sweep we pool *all* verbalized belief predictions across the four game configurations and compute one reliability curve + ECE + lift table. Because each config has a different impostor prior (4C_1I → 0.25, 4C_2I → 0.50, 5C_1I → 0.20, 5C_2I → 0.40), the *per-model prior* shown in the table is the empirical mean of the per-config priors weighted by sample size — i.e. the base rate the model would hit by random guessing across the pooled sample.

In [ ]:
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from evaluations.metrics_calculator import (
    EpistemicLogParser,
    _extract_ground_truth_from_summary,
    _load_all_summaries,
)

# ── 1. Build experiment_dir -> model map from the latest sweep CSV ──────
sweep_csvs = sorted(glob.glob(
    os.path.join(RESULTS_PATH, "*_tom_metrics_sweep.csv")
))
exp_to_model: "dict[str, str]" = {}
if sweep_csvs:
    latest_csv = sweep_csvs[-1]
    try:
        tom_df = pd.read_csv(latest_csv, usecols=["experiment", "model"])
        exp_to_model = (
            tom_df.dropna(subset=["experiment", "model"])
            .drop_duplicates(subset=["experiment"])
            .set_index("experiment")["model"]
            .to_dict()
        )
        print(f"Loaded model labels for {len(exp_to_model)} experiments "
              f"from {os.path.basename(latest_csv)}")
    except Exception as e:
        print(f"Could not parse {latest_csv}: {e}")
else:
    print(f"No *_tom_metrics_sweep.csv found in {RESULTS_PATH}; "
          "cannot map experiment dirs to models.")

# ── 2. Aggregate (predicted_prob, ground_truth) per MODEL ──────────────
# Two stores per model: one for verbalized beliefs (always available),
# one for logprob-derived beliefs (only when the backend reports
# top_logprobs — basically vLLM and OpenAI).
per_model: "dict[str, dict[str, list]]" = {}

epi_files = sorted(glob.glob(os.path.join(LOGS_PATH, "*", "epistemic-states.jsonl")))
print(f"Found {len(epi_files)} epistemic-state files.")

for epi_path in epi_files:
    expt_dir = os.path.dirname(epi_path)
    expt_name = os.path.basename(expt_dir)
    model_key = exp_to_model.get(expt_name)
    if model_key is None:
        continue

    summary_path = os.path.join(expt_dir, "summary.json")
    summaries = _load_all_summaries(summary_path)
    if not summaries:
        continue

    impostors_by_game: "dict[str, set[str]]" = {}
    prior_by_game: "dict[str, float]" = {}
    for summary in summaries:
        game_key = next(
            (k for k in summary if isinstance(k, str) and k.startswith("Game")),
            None,
        )
        if not game_key:
            continue
        gt = _extract_ground_truth_from_summary(summary, game_key)
        if gt is None:
            continue
        impostors_by_game[game_key] = gt.impostor_names
        cfg = summary[game_key].get("config", {}) or {}
        np_, ni_ = cfg.get("num_players"), cfg.get("num_impostors")
        if np_ and ni_:
            n_living_opps = max(1, int(np_) - 1)
            prior_by_game[game_key] = float(ni_) / float(n_living_opps)

    if not impostors_by_game:
        continue

    union_impostors: "set[str]" = set().union(*impostors_by_game.values())

    parser = EpistemicLogParser()
    snapshots = parser.parse_file(epi_path)

    bucket = per_model.setdefault(
        model_key,
        {"verb_p": [], "verb_y": [], "lp_p": [], "lp_y": []},
    )

    for s in snapshots:
        gi = s.game_index
        impostor_set = (
            impostors_by_game.get(gi)
            if gi and gi in impostors_by_game
            else union_impostors
        )

        for p_name, vp in (s.belief_distribution or {}).items():
            y = 1.0 if p_name in impostor_set else 0.0
            bucket["verb_p"].append(float(vp))
            bucket["verb_y"].append(y)

        for p_name, lpp in (s.belief_distribution_logprobs or {}).items():
            y = 1.0 if p_name in impostor_set else 0.0
            bucket["lp_p"].append(float(lpp))
            bucket["lp_y"].append(y)

for bucket in per_model.values():
    for k in bucket:
        bucket[k] = np.asarray(bucket[k], dtype=float)

print("\nVerbalized predictions per model:")
for m, b in per_model.items():
    print(f"  {m:20s} {len(b['verb_p']):>6d} verb,  {len(b['lp_p']):>6d} lp")

In [ ]:
# ── 3. Reusable calibration helpers (mirrors cell 34) ──────────────────
def _reliability_per_model(p, y, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(p, bins, right=False) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        rows.append((p[mask].mean(), y[mask].mean(), mask.sum() / len(p)))
    return rows


def _ece_per_model(p, y, n_bins=10):
    if len(p) == 0:
        return float("nan")
    return float(sum(abs(mp - my) * w
                     for mp, my, w in _reliability_per_model(p, y, n_bins)))


def _top_decile_lift_per_model(p, y, threshold=0.9):
    mask = p >= threshold
    if mask.sum() == 0:
        return float("nan"), 0, float("nan")
    prec = float(y[mask].mean())
    base = float(y.mean()) if len(y) else float("nan")
    lift = prec / base if base else float("nan")
    return lift, int(mask.sum()), prec


# ── 4. Combined per-model reliability plots (one panel per signal) ─────
MODEL_ORDER = [m for m in MODELS_TO_RUN if m in per_model] + \
    [m for m in per_model if m not in MODELS_TO_RUN]

CMAP = plt.get_cmap("tab10")
COLOR = {m: CMAP(i % 10) for i, m in enumerate(MODEL_ORDER)}


def _plot_combined(signal_key, title, fname):
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")

    legend_labels = []
    for m in MODEL_ORDER:
        b = per_model.get(m, {})
        p = b.get(f"{signal_key}_p", np.array([]))
        y = b.get(f"{signal_key}_y", np.array([]))
        if len(p) == 0:
            continue
        rows = _reliability_per_model(p, y, n_bins=10)
        if not rows:
            continue
        xs = [mp for mp, _, _ in rows]
        ys = [my for _, my, _ in rows]
        ax.plot(xs, ys, marker="o", lw=2, color=COLOR[m], label=None)
        prior = float(y.mean())
        ece = _ece_per_model(p, y)
        legend_labels.append(
            f"{m}  (n={len(p)}, prior={prior:.2f}, ECE={ece:.3f})"
        )
        ax.scatter([prior], [prior], color=COLOR[m], marker="x", s=80, zorder=5)

    handles, _ = ax.get_legend_handles_labels()
    proxies = [plt.Line2D([0], [0], color=COLOR[m], marker="o", lw=2)
               for m in MODEL_ORDER if m in per_model
               and len(per_model[m].get(f"{signal_key}_p", [])) > 0]
    ax.legend(
        [handles[0]] + proxies,
        ["perfect calibration"] + legend_labels,
        loc="upper left", fontsize=9, framealpha=0.9,
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted P(opponent is Impostor)")
    ax.set_ylabel("Observed frequency of Impostor")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    out_png = os.path.join(RESULTS_PATH, fname)
    fig.savefig(out_png, dpi=150)
    plt.show()
    print(f"Saved figure to: {out_png}")


_plot_combined(
    "verb",
    "Verbalized belief calibration — per model (all configs pooled)",
    "belief_calibration_verbalized_per_model.png",
)
_plot_combined(
    "lp",
    "Logprob-derived belief calibration — per model (all configs pooled)",
    "belief_calibration_logprob_per_model.png",
)

In [ ]:
# ── 5. Per-model lift table (pooled across configs) ───────────────────
rows = []
for m in MODEL_ORDER:
    b = per_model.get(m, {})

    def _stats(p, y):
        if len(p) == 0:
            return dict(n=0, prior=float("nan"), ece=float("nan"),
                        prec=float("nan"), lift=float("nan"), n_top=0)
        lift, n_top, prec = _top_decile_lift_per_model(p, y)
        return dict(
            n=int(len(p)),
            prior=float(y.mean()),
            ece=_ece_per_model(p, y),
            prec=prec,
            lift=lift,
            n_top=n_top,
        )

    v = _stats(b.get("verb_p", np.array([])), b.get("verb_y", np.array([])))
    l = _stats(b.get("lp_p",   np.array([])), b.get("lp_y",   np.array([])))

    rows.append({
        "model":                m,
        "verb_n":               v["n"],
        "verb_prior":           v["prior"],
        "verb_ECE":             v["ece"],
        "verb_precision@p\u22650.9": v["prec"],
        "verb_lift@p\u22650.9":     v["lift"],
        "verb_n_top":           v["n_top"],
        "logp_n":               l["n"],
        "logp_prior":           l["prior"],
        "logp_ECE":             l["ece"],
        "logp_precision@p\u22650.9": l["prec"],
        "logp_lift@p\u22650.9":     l["lift"],
        "logp_n_top":           l["n_top"],
    })

per_model_lift_df = pd.DataFrame(rows).set_index("model")
fmt = {
    "verb_prior":               "{:.2f}",
    "verb_ECE":                 "{:.3f}",
    "verb_precision\u22650.9":  "{:.2f}",
    "verb_lift\u22650.9":       "{:.2f}",
    "logp_prior":               "{:.2f}",
    "logp_ECE":                 "{:.3f}",
    "logp_precision\u22650.9":  "{:.2f}",
    "logp_lift\u22650.9":       "{:.2f}",
}

print("\nPer-model belief calibration  (all configs pooled, UNPAIRED)")
print("-" * 60)
try:
    from IPython.display import display
    display(per_model_lift_df.style.format(fmt).set_caption(
        "prior = empirical P(opponent is Impostor) for the pooled sample "
        "(weighted average over the 4 configs). "
        "ECE = expected calibration error: sample-weighted mean gap "
        "between predicted probability and observed frequency across 10 "
        "bins; 0 = perfectly calibrated. "
        "precision@p\u22650.9 = fraction of opponents the model flagged "
        "with probability \u2265 0.9 who really were Impostors. "
        "lift@p\u22650.9 = precision@p\u22650.9 / prior; >1 means "
        "confident calls beat random chance for this model."
    ))
except Exception:
    print(per_model_lift_df.to_string(
        formatters={k: v.format for k, v in fmt.items()}
    ))

out_csv = os.path.join(RESULTS_PATH, "belief_calibration_by_model.csv")
per_model_lift_df.to_csv(out_csv)
print(f"\nSaved table to: {out_csv}")